# Fabric migration (v2) – step 6: restore the data

Run after step 5 (`V2_5_Restore_From_Git`) has brought the items back (from Git, or from the raw copy if Azure DevOps was unavailable).
Git holds only definitions, so the restored Lakehouses, Warehouses and KQL databases are **empty**. This notebook
loads their data from the backup run that the snapshot recorded (`SECONDARY_BACKUP_RUN_ID`, written by
`V2_2_Backup_Data` after the freeze) and proves it arrived.

| Item | Saved in ADLS by the backup | Restored by this notebook | Checked by |
|---|---|---|---|
| Lakehouse | `Files/` and `Tables/` (Delta; with schemas `Tables/<schema>/<table>`) | file copy into OneLake, Delta logs last; only into a Lakehouse with the same layout (schemas or not) | every file present with the same size |
| KQL database | Parquet export per table + schema script | `.ingest` per table (update policies paused) | row count = exported rows |
| Warehouse | `Tables/` (Delta) | `COPY INTO`, or typed physical-column `OPENROWSET` + `INSERT` for mapped tables; script or run here (`WAREHOUSE_LOAD = "run"`) | row count and available column NULL counts match the Delta log |
| SQL analytics endpoint | views, functions, procedures | a script, or run here (`SQL_OBJECTS = "run"`) | objects present |

**How it protects the data**
- Reads the manifest (hash-checked) and only the backup run it names; refuses items whose backup step wasn't OK.
- Finds each workspace where V2_5 restored it (else by ID, else by name) and each restored item by type and name;
  the IDs in the backup are not trusted.
- Never overwrites or deletes: files are created with *if-none-match*, a Delta table or KQL table that already
  holds data in the target is left alone and reported, Warehouse tables are loaded only when empty.
- Dry run first (plan + `DATA_ID`); a real run needs `CONFIRM_DATA_ID`. Progress is saved in ADLS, so a rerun
  continues where it stopped and never loads a table twice.

**Requirements**
- Contributor or higher on the target workspaces; Storage Blob Data Reader on the backup container (the Eventhouse
  and the Warehouse read the backup files with your identity).

In [ ]:
SNAPSHOT_ID = "<snapshot id printed by V2_3_Git_Snapshot>"
# Each workspace is found where V2_5 restored it (restore_state.json), else by ID, else by name.

ADLS_ACCOUNT = "<storage-account-name>"
ADLS_CONTAINER = "fabric-backup"
ADLS_ROOT = "fabric-backups"

# Step 5 (V2_5) must have finished for a workspace before its data is loaded (from Git or from ADLS, both count).
# False only if the items were recreated some other way (by hand).
REQUIRE_GIT_RESTORE_DONE = True

RESTORE_LAKEHOUSE_DATA = True
RESTORE_KQL_DATA = True
# Where the KQL rows come from (V2_2 KQL_BACKUP_MODE): "auto" = the Parquet files if they were exported OK, else
# the Azure Data Explorer copy (Kusto Copy); "parquet" or "adx" = only that one. The ADX copy is read by the new
# Eventhouse with YOUR identity (cross-cluster query): you need Viewer on the ADX database.
KQL_RESTORE_SOURCE = "auto"
# Warehouse tables and SQL endpoint objects: "script" = write a .sql file to ADLS to run in the SQL editor,
# "run" = also run it here through Fabric's SQL connector with your identity, "off" = skip.
WAREHOUSE_LOAD = "script"
SQL_OBJECTS = "script"
# Lakehouse files: "fastcp" = bulk copy of each table folder (data first, _delta_log last) with azcopy
# (notebookutils.fs.fastcp), much faster for GBs; any file it doesn't place is copied one by one. "per_file" = every
# file through this notebook only. Both copy the backup byte for byte (Delta history kept) and never replace a file
# that is already there; the backup in ADLS is never moved or changed.
LAKEHOUSE_COPY_METHOD = "fastcp"
MAX_PARALLEL_FILES = 8                     # per_file copy (and the fallback)

# 1. DRY_RUN = True: prints the plan and a DATA ID. 2. DRY_RUN = False with CONFIRM_DATA_ID = that ID.
DRY_RUN = True
CONFIRM_DATA_ID = ""

# ---------------------------------------------------------------- rarely needed
# Only if a workspace was recreated under ANOTHER name: {"source workspace ID (manifest)": "workspace ID now"}.
WORKSPACE_MAP = {}

## Helpers
Same helpers as the other v2 notebooks, plus OneLake, Kusto and SQL loaders. Nothing runs here.

In [ ]:
BUILD_ID = "b4da7cb88cdb"
print("V2_6_Restore_Data build b4da7cb88cdb (compare with BUILD_IDS.txt of the package you meant to run)")
"""Credential redaction for diagnostic output, never for API or backup payloads."""
import re
from collections.abc import Mapping
from typing import TypeVar, overload
from urllib.parse import quote

REDACTED = "<redacted>"
if "_REDACTION_VALUES" not in globals():
    _REDACTION_VALUES: set[str] = set()

_REDACTION_KEYS = {
    "authorization", "accesstoken", "refreshtoken", "clientsecret", "serviceprincipalsecret",
    "password", "passwd", "accountkey", "apikey", "personalaccesstoken", "pat", "secret", "token",
    "connectionstring", "sastoken", "sharedaccesssignature", "sig",
}
_REDACTION_PATTERNS = (
    re.compile(r"\bgh[pousr]_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{40,}\b"),
    re.compile(r"\beyJ[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\b"),
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----.*?"
               r"-----END (?:RSA |EC |OPENSSH )?PRIVATE KEY-----", re.DOTALL),
)
_REDACTION_AUTH = re.compile(r"\b(Bearer|Basic)([ \t]+)[A-Za-z0-9_+/=.-]{16,}", re.IGNORECASE)
_REDACTION_QUERY = re.compile(
    r"([?&](?:sig|access_token|refresh_token|client_secret|password|api_key)=)[^&\s\"'<>]+",
    re.IGNORECASE,
)
_REDACTION_ASSIGNMENT = re.compile(
    r"""(\b(?:client[_-]?secret|servicePrincipalSecret|access[_-]?token|refresh[_-]?token|
    password|passwd|account[_-]?key|api[_-]?key|personalAccessToken|sas[_-]?token|pat|secret|token)
    \b["']?\s*[:=]\s*["']?)([^\s"'&,;}\]<>]+)""",
    re.IGNORECASE | re.VERBOSE,
)


def register_sensitive_value(value: str) -> None:
    if len(value) >= 8:
        _REDACTION_VALUES.add(value)


def redact_sensitive_text(text: str) -> str:
    for value in sorted(_REDACTION_VALUES.copy(), key=len, reverse=True):
        text = text.replace(value, REDACTED).replace(quote(value, safe=""), REDACTED)
    for pattern in _REDACTION_PATTERNS:
        text = pattern.sub(REDACTED, text)
    text = _REDACTION_AUTH.sub(lambda m: m[1] + m[2] + REDACTED, text)
    text = _REDACTION_QUERY.sub(lambda m: m[1] + REDACTED, text)
    return _REDACTION_ASSIGNMENT.sub(lambda m: m[1] + REDACTED, text)


_Key = TypeVar("_Key")


@overload
def redact_sensitive_data(value: str) -> str: ...


@overload
def redact_sensitive_data(value: Mapping[_Key, object]) -> dict[_Key, object]: ...


@overload
def redact_sensitive_data(value: list[object]) -> list[object]: ...


@overload
def redact_sensitive_data(value: tuple[object, ...]) -> tuple[object, ...]: ...


@overload
def redact_sensitive_data(value: object) -> object: ...


def redact_sensitive_data(value: object) -> object:
    if isinstance(value, Mapping):
        out = {}
        for key, item in value.items():
            normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
            if normalized in _REDACTION_KEYS and item is not None and not (isinstance(item, str) and item == ""):
                out[key] = REDACTED
            else:
                out[key] = redact_sensitive_data(item)
        return out
    if isinstance(value, list):
        return [redact_sensitive_data(item) for item in value]
    if isinstance(value, tuple):
        return tuple(redact_sensitive_data(item) for item in value)
    return redact_sensitive_text(value) if isinstance(value, str) else value
import base64, datetime as dt, hashlib, json, re, threading, time, uuid
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote

import pandas as pd

pd.set_option("display.max_colwidth", None)   # never cut error messages or fixes in tables
import requests

try:
    notebookutils  # preloaded in Fabric notebooks
except NameError:
    import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
STORAGE_VERSION = "2023-11-03"
MAX_ATTEMPTS = 6
MAX_RETRY_WAIT_SECONDS = 15 * 60
LRO_TIMEOUT_SECONDS = 60 * 60
CHUNK_BYTES = 8 * 1024 * 1024
ZERO_GUID = "00000000-0000-0000-0000-000000000000"
GUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
NOW_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RESULTS = []           # one row per item and step: this notebook's log
_lock = threading.Lock()
CALLS, RETRIES = Counter(), Counter()


def norm_id(value):
    text = str(value or "").strip().strip("{}").lower()
    return text if GUID_RE.match(text) else None


def safe_name(name, max_len=80):
    """A folder-safe version of a Fabric display name."""
    text = re.sub(r'[\\/:*?"<>|#%\x00-\x1f]', "_", str(name or "")).strip().strip(".")
    return (text[:max_len].rstrip(" .") or "unnamed")


def log(workspace, item, step, status, detail=""):
    """One row of this notebook's log. workspace: name; item: dict with type/displayName/id (or {})."""
    detail = redact_sensitive_text(str(detail))
    with _lock:
        RESULTS.append({"Workspace": workspace, "Item": item.get("displayName", ""), "Type": item.get("type", ""),
                        "ItemId": item.get("id", ""), "Step": step, "Status": status,
                        "Detail": str(detail)})
    if status not in ("OK", "Dry run", "Skipped", "Unchanged", "Info", "Ready"):
        print(f"  ! {workspace} / {item.get('type', '')} {item.get('displayName', '')} / {step}: {status} - "
              f"{detail}")


def show(df, rows=200):
    try:
        display(df.head(rows))
    except NameError:
        print(df.head(rows).to_string())


def human(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


class HttpError(Exception):
    def __init__(self, status, code, message, url):
        self.status, self.code, self.message, self.url = status, code or "", redact_sensitive_text(str(message or "")), url
        super().__init__(self.short())

    def short(self):
        head = f"HTTP {self.status} {self.code}".strip() if self.status else str(self.code)
        return f"{head}: {self.message}" if self.message else head


RAW_BODY_LIMIT = 4000   # only for non-JSON bodies (for example a whole HTML page); JSON error messages are kept whole


def _raw_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")              # HTML page -> its text
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= RAW_BODY_LIMIT:
        return text
    return text[:RAW_BODY_LIMIT] + f" ... [{len(text) - RAW_BODY_LIMIT} more characters of the response body]"


def _error_details(resp):
    try:
        body = resp.json()
    except ValueError:
        return "", _raw_text(resp.text)
    if not isinstance(body, dict):
        return "", json.dumps(body)
    err = body.get("error") if isinstance(body.get("error"), dict) else body
    code = err.get("errorCode") or err.get("code") or err.get("typeKey") or err.get("error") or ""
    msg = err.get("message") or err.get("@message") or err.get("error_description") or ""
    if not msg:
        msg = json.dumps(body)                                  # unknown error shape: keep all of it
    return str(code), str(msg)


def _retry_after(resp, attempt):
    try:
        return max(1, int(float(resp.headers.get("Retry-After", ""))))
    except (TypeError, ValueError):
        return min(300, 10 * 2 ** (attempt - 1))


_local = threading.local()


def _session():
    if getattr(_local, "session", None) is None:
        _local.session = requests.Session()
    return _local.session


def http(method, url, audience=None, *, auth=None, headers=None, params=None, json_body=None, data=None,
         timeout=300, retry=True, label="other"):
    """One HTTP call with a fresh Entra token. Retries 408/429/5xx (honouring Retry-After) and network errors."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        hdrs = dict(headers or {})
        if auth:
            hdrs["Authorization"] = auth
        elif audience:
            hdrs["Authorization"] = "Bearer " + notebookutils.credentials.getToken(audience)
        if hdrs.get("Authorization"):
            register_sensitive_value(hdrs["Authorization"].split(" ", 1)[-1])
        try:
            resp = _session().request(method, url, params=params, json=json_body, data=data,
                                      headers=hdrs, timeout=timeout)
        except requests.RequestException as exc:
            if not retry or attempt == MAX_ATTEMPTS:
                raise HttpError(None, type(exc).__name__, str(exc), url)
            time.sleep(min(60, 5 * attempt))
            continue
        with _lock:
            CALLS[label] += 1
        if retry and (resp.status_code in (408, 429) or resp.status_code >= 500):
            wait = _retry_after(resp, attempt)
            if attempt < MAX_ATTEMPTS and wait <= MAX_RETRY_WAIT_SECONDS:
                with _lock:
                    RETRIES[label] += 1
                time.sleep(wait)
                continue
        if resp.status_code >= 400:
            raise HttpError(resp.status_code, *_error_details(resp), url)
        return resp


# ---------------------------------------------------------------- Fabric REST API
def _wait_operation(resp, want_result=True):
    """Follow a Fabric long-running operation (202 Accepted) to its result."""
    location = resp.headers.get("Location")
    op_id = resp.headers.get("x-ms-operation-id")
    if not location and op_id:
        location = f"{FABRIC_API}/operations/{op_id}"
    if not location:
        return {}
    wait = _retry_after(resp, 1) if resp.headers.get("Retry-After") else 2
    deadline = time.monotonic() + LRO_TIMEOUT_SECONDS
    while True:
        time.sleep(min(max(wait, 1), 30))
        state = http("GET", location, "pbi", label="fabric").json()
        status = str(state.get("status", "")).lower()
        if status == "succeeded":
            if not want_result:
                return state
            try:
                result = http("GET", location.rstrip("/") + "/result", "pbi", label="fabric")
            except HttpError as exc:
                if exc.status in (400, 404):      # operations without a result (commit, update from Git)
                    return state
                raise
            return result.json() if result.content else {}
        if status in ("failed", "cancelled", "canceled"):
            err = state.get("error") or {}
            raise HttpError(None, err.get("errorCode") or status, err.get("message"), location)
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", "the long-running operation didn't finish", location)
        wait = 3


def fabric(method, path, body=None, params=None, want_result=True):
    resp = http(method, FABRIC_API + path, "pbi", json_body=body, params=params, label="fabric")
    if resp.status_code == 202:
        return _wait_operation(resp, want_result)
    return resp.json() if resp.content else {}


def fabric_pages(path, key="value", params=None):
    url, prm, seen = FABRIC_API + path, dict(params or {}), set()
    while url:
        body = http("GET", url, "pbi", params=prm or None, label="fabric").json()
        yield from body.get(key) or []
        if body.get("continuationUri"):
            url, prm = body["continuationUri"], {}
        elif body.get("continuationToken"):
            url, prm = FABRIC_API + path, {**(params or {}), "continuationToken": body["continuationToken"]}
        else:
            break
        marker = (url, json.dumps(prm, sort_keys=True))
        if marker in seen:
            break
        seen.add(marker)


# ---------------------------------------------------------------- ADLS Gen2 (secondary backup + migration records)
ADLS_HEADERS = {"x-ms-version": STORAGE_VERSION}


def adls_url(path):
    return f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}/{quote(path.strip('/'), safe='/')}"


def adls_create(path):
    http("PUT", adls_url(path), "storage", headers=ADLS_HEADERS, params={"resource": "file"}, label="adls")


def adls_append(path, position, data):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=data,
         params={"action": "append", "position": str(position)}, label="adls")


def adls_flush(path, position):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=b"",
         params={"action": "flush", "position": str(position)}, label="adls")


def adls_write(path, data):
    """Write a whole (small) file, replacing any existing one."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    adls_create(path)
    for pos in range(0, len(data), CHUNK_BYTES):
        adls_append(path, pos, data[pos:pos + CHUNK_BYTES])
    adls_flush(path, len(data))
    return len(data)


def adls_list(path, recursive=True):
    """[(path relative to `path`, size, is_dir)] under a folder of the container; [] if it doesn't exist."""
    url = f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}"
    params = {"resource": "filesystem", "recursive": str(recursive).lower(), "directory": path.strip("/")}
    out, prefix = [], path.strip("/") + "/"
    while True:
        try:
            resp = http("GET", url, "storage", headers=ADLS_HEADERS, params=params, label="adls")
        except HttpError as exc:
            if exc.status == 404:
                return out
            raise
        for e in resp.json().get("paths", []):
            name = e["name"][len(prefix):] if e["name"].startswith(prefix) else e["name"]
            out.append((name, int(e.get("contentLength") or 0), str(e.get("isDirectory", "false")).lower() == "true"))
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return out
        params = {**params, "continuation": token}


def adls_read(path):
    return http("GET", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls").content


def adls_delete(path):
    http("DELETE", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")


def adls_exists(path):
    try:
        http("HEAD", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")
        return True
    except HttpError as exc:
        if exc.status == 404:
            return False
        raise


# ---------------------------------------------------------------- Power BI item types
# Power BI items survive a cross-region move; every other item type is a Fabric item that must be deleted first.
POWER_BI_TYPES = {"Report", "SemanticModel", "Dashboard", "PaginatedReport"}
# Items Fabric creates and deletes together with their parent; they are never saved in Git on their own.
CHILD_TYPES = {"SQLEndpoint"}
DATA_TYPES = {"Lakehouse": "Data", "Warehouse": "Data", "KQLDatabase": "KQL export"}
# Dataflow Gen2 staging items: Fabric creates them for Dataflows Gen2 (one Lakehouse and one Warehouse per workspace,
# named StagingLakehouseForDataflows_<timestamp> / StagingWarehouseForDataflows_<timestamp>) and creates them again
# when a Dataflow Gen2 runs. Not saved in Git, not backed up, not restored; V2_4 still deletes them (a Fabric item
# left behind blocks a tenant migration).
STAGING_RE = re.compile(r"^Staging(Lakehouse|Warehouse)ForDataflows(_\d+)?$", re.IGNORECASE)


def is_dataflow_staging(item):
    return bool(item.get("staging")) or (item.get("type") in ("Lakehouse", "Warehouse", "SQLEndpoint")
                                         and bool(STAGING_RE.match(str(item.get("displayName") or ""))))
KQL_ADX_STEP = "Kusto Copy"      # V2_2 with KQL_BACKUP_MODE "adx"/"both": the KQL rows are in Azure Data Explorer


def data_step(item_type, status_of):
    """(backup step, status) holding an item's data. KQL databases: the Parquet export if it's OK, otherwise the
    Kusto Copy to ADX if that's OK, otherwise whichever the backup recorded. status_of(step) -> status or ""."""
    step = DATA_TYPES.get(item_type)
    if item_type != "KQLDatabase":
        return step, status_of(step) if step else ""
    exp, adx = status_of(step), status_of(KQL_ADX_STEP)
    if exp == "OK" or not adx:
        return (step, exp) if exp or not adx else (KQL_ADX_STEP, adx)
    if adx == "OK" or not exp:
        return KQL_ADX_STEP, adx
    return step, exp


GUID_ANY_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")


def definition_hash(parts):
    """Fingerprint of an item definition (every part except .platform), to prove it didn't change."""
    h = hashlib.sha256()
    for p in sorted(parts, key=lambda x: x["path"]):
        if p["path"] == ".platform":
            continue
        h.update(p["path"].encode("utf-8") + b"\0" + base64.b64decode(p.get("payload") or "") + b"\0")
    return h.hexdigest()


def adls_only(item):
    """Item in a Git-linked workspace that Git doesn't hold: V2_5 recreates it from its ADLS definition.
    ("secondary backup ..." is the label older snapshots used.)"""
    rec = str(item.get("recovery") or "")
    return not item.get("inGit") and (rec.startswith("ADLS definition (V2_5") or rec.startswith("secondary backup"))


def key_of(item_type, name):
    return f"{item_type}|{str(name).strip().lower()}"


# ---------------------------------------------------------------- Azure DevOps (read the snapshot, manage branches)
ADO_API_VERSION = "7.1"
_ado_auth = {}


ADO_RESOURCE = "499b84ac-1321-427f-aa17-267ca6975798"     # Azure DevOps' fixed Entra application ID


def ado_mode():
    mode = str(globals().get("ADO_AUTH", "pat") or "pat").strip().lower().replace("-", "_").replace(" ", "_")
    if mode in ("sp", "spn", "serviceprincipal"):
        mode = "service_principal"
    if mode not in ("pat", "service_principal"):
        raise ValueError(f"ADO_AUTH = {globals().get('ADO_AUTH')!r}: use \"pat\" or \"service_principal\"")
    return mode


def ado_auth_missing():
    """Settings the chosen ADO_AUTH needs that are still empty or placeholders."""
    need = ["ADO_PAT_SECRET_NAME"] if ado_mode() == "pat" else ["ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME"]
    return [n for n in need if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def _clean_secret(value):
    value = str(value or "").strip().strip('"').strip("'").strip()
    register_sensitive_value(value)
    return value


_sp_auth = {}


def sp_header():
    """Entra token (Bearer header) of the service principal in ADO_SP_* for Azure DevOps, renewed 5 minutes before it
    expires. Used by ADO_AUTH = "service_principal" and to check GIT_LINK_AUTH = "service_principal"."""
    if "header" in _sp_auth and _sp_auth.get("expires", 0) > time.time() + 300:
        return _sp_auth["header"]
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    kind, _ = secret_shape(secret)
    _sp_auth.update(secret_length=len(secret), secret_kind=kind)
    url = f"https://login.microsoftonline.com/{quote(str(ADO_SP_TENANT_ID).strip())}/oauth2/v2.0/token"
    try:
        tok = http("POST", url, data={"grant_type": "client_credentials", "client_id": str(ADO_SP_CLIENT_ID).strip(),
                                      "client_secret": secret, "scope": ADO_RESOURCE + "/.default"},
                   label="entra").json()
    except HttpError as exc:
        raise HttpError(exc.status, exc.code or "SignInFailed",
                        f"service principal sign-in failed: {exc.message} {sp_signin_fix(exc.message, kind)}".strip(),
                        url) from None
    _sp_auth["header"] = "Bearer " + tok["access_token"]
    register_sensitive_value(tok["access_token"])
    _sp_auth["expires"] = time.time() + int(tok.get("expires_in", 3600))
    return _sp_auth["header"]


def ado_header():
    """Authorization header for Azure DevOps: a personal access token, or an Entra token of a service principal
    (renewed 5 minutes before it expires, so long runs keep working). Secrets come from Key Vault and are never shown."""
    if ado_mode() == "pat":
        if "header" not in _ado_auth:
            pat = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_PAT_SECRET_NAME))
            _ado_auth.update(mode="pat", pat_length=len(pat), pat_shape=secret_shape(pat))
            _ado_auth["header"] = "Basic " + base64.b64encode((":" + pat).encode()).decode()
        return _ado_auth["header"]
    _ado_auth["mode"] = "service_principal"
    try:
        header = sp_header()
    finally:
        _ado_auth.update(pat_length=_sp_auth.get("secret_length"), sp_secret_kind=_sp_auth.get("secret_kind"))
    _ado_auth.update(header=header, expires=_sp_auth["expires"])
    return header


def sp_signin_fix(message, kind=""):
    m = str(message or "")
    if kind == "pat":
        return ("The Key Vault secret looks like a personal access token, not a client secret: set ADO_AUTH = \"pat\" "
                "or store the app's client secret value.")
    if "AADSTS7000215" in m or "AADSTS7000222" in m:
        return ("Fix: the client secret is wrong or expired. Store the secret's Value (not its Secret ID) from the app "
                "registration > Certificates & secrets, as a new version of the Key Vault secret.")
    if "AADSTS700016" in m:
        return "Fix: ADO_SP_CLIENT_ID isn't an app in tenant ADO_SP_TENANT_ID: check both IDs."
    if "AADSTS90002" in m or "AADSTS900023" in m:
        return "Fix: ADO_SP_TENANT_ID is wrong: use the Directory (tenant) ID from the app registration's overview."
    return "Check ADO_SP_TENANT_ID, ADO_SP_CLIENT_ID and the client secret value."


def secret_shape(value):
    """(kind, hint) describing what a secret looks like, without revealing it. kind 'pat' = plausible DevOps token."""
    v, n = value, len(value)
    if not v:
        return "empty", "The secret is empty: paste the token as the secret value."
    if re.fullmatch(r"[a-z2-7]{52}", v) or (n == 84 and re.fullmatch(r"[A-Za-z0-9]{84}", v)):
        return "pat", ""
    if re.match(r"^[A-Za-z0-9_\-.~]{3}\dQ~", v) or (n == 40 and "~" in v):
        return "client secret", ("It looks like an Entra ID app registration client secret (40 characters with 'Q~'), "
                                 "not an Azure DevOps personal access token. Create a token in Azure DevOps (User "
                                 "settings > Personal access tokens, scope Code: Read & write) and store that value.")
    if re.fullmatch(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", v):
        return "GUID", ("It is a GUID: probably a client ID, tenant ID or a client secret's 'Secret ID'. None of "
                        "these is an Azure DevOps token.")
    if v.startswith(("ghp_", "gho_", "ghs_", "github_pat_")):
        return "GitHub token", "It looks like a GitHub token, not an Azure DevOps personal access token."
    if re.fullmatch(r"[0-9a-fA-F]{40}", v):
        return "40 hex", ("It is 40 hexadecimal characters: a Git commit ID or an old GitHub token, not an Azure "
                          "DevOps personal access token.")
    if v.startswith(("{", "[")):
        return "JSON", "It holds a JSON value: store only the token itself."
    if v.lower().startswith(("http://", "https://")):
        return "URL", "It holds a URL: store the token, and put the organization in ADO_ORGANIZATION."
    if v.lower().startswith(("bearer ", "basic ")) or re.search(r"\s", v):
        return "several values", "It holds more than one value (spaces or a 'Bearer'/'Basic' prefix): store only the token."
    if v.startswith("eyJ"):
        return "JWT", "It is an Entra access token (JWT), which expires within hours: store a personal access token."
    return "unknown", (f"Azure DevOps tokens have 52 or 84 characters; this has {n}. Check that the secret holds "
                       "the token and nothing else.")


def ado_org_name(value):
    """'contoso', 'https://dev.azure.com/contoso/...' or 'https://contoso.visualstudio.com' -> 'contoso'."""
    text = str(value or "").strip().rstrip("/")
    m = re.match(r"^(?:https?://)?dev\.azure\.com/([^/]+)", text, re.I) or \
        re.match(r"^(?:https?://)?([^./]+)\.visualstudio\.com", text, re.I)
    return m.group(1) if m else text


SIGN_IN_PAGE = ("Azure DevOps answered with its sign-in page instead of data (HTTP 203): it didn't accept the "
                "personal access token for this organization")


SP_NOT_IN_ORG = ("Add the service principal to the organization (Organization settings > Users > Add users, "
                 "search the app name, Basic access) and to the project's Contributors group; the organization must "
                 "be connected to the app's Entra tenant.")


def ado_call(url, method="GET", params=None, body=None, raw=False, auth=None):
    """auth = another Authorization header (for example sp_header()); default: the ADO_AUTH sign-in."""
    sp = auth is not None or _ado_auth.get("mode") == "service_principal"
    try:
        resp = http(method, url, auth=auth or ado_header(), params={"api-version": ADO_API_VERSION, **(params or {})},
                    json_body=body, label="devops", headers={"Accept": "application/octet-stream"} if raw else None)
    except HttpError as exc:
        if exc.status == 401 and (sp or _ado_auth.get("mode") == "service_principal"):
            raise HttpError(401, exc.code or "Unauthorized", f"{exc.message} {SP_NOT_IN_ORG}".strip(), url) from None
        raise
    ctype = resp.headers.get("Content-Type", "json")
    if resp.status_code == 203 or "html" in ctype:
        if sp or _ado_auth.get("mode") == "service_principal":
            raise HttpError(203, "SignInPage", "Azure DevOps answered with its sign-in page instead of data (HTTP "
                            "203): it didn't accept the service principal's token. " + SP_NOT_IN_ORG, url)
        kind, hint = _ado_auth.get("pat_shape", ("pat", ""))
        raise HttpError(203, "SignInPage", SIGN_IN_PAGE + (f". {hint}" if kind != "pat" else ""), url)
    if raw:
        return resp.content
    if "json" not in ctype:
        raise HttpError(resp.status_code, "NotJson", f"Azure DevOps returned {ctype}, not JSON", url)
    return resp.json() if resp.content else {}


def ado(gp, method, path, params=None, body=None, raw=False):
    """Call the Azure DevOps Git API of the repository in a Fabric Git connection (gp = gitProviderDetails)."""
    url = (f"https://dev.azure.com/{quote(ado_org_name(gp['organizationName']))}/{quote(gp['projectName'])}"
           f"/_apis/git/repositories/{quote(gp['repositoryName'])}{path}")
    return ado_call(url, method, params, body, raw)


def ado_ref(gp, branch):
    """{'objectId', 'isLocked'} of a branch, or None if it doesn't exist."""
    refs = ado(gp, "GET", "/refs", params={"filter": f"heads/{branch}"}).get("value", [])
    return next((r for r in refs if r.get("name") == f"refs/heads/{branch}"), None)


def ado_create_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": "0" * 40,
                                          "newObjectId": commit}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"),
                        f"couldn't create branch {branch}", branch)


def ado_lock(gp, branch, locked=True):
    ado(gp, "PATCH", "/refs", params={"filter": f"heads/{branch}"}, body={"isLocked": locked})


def ado_delete_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": commit,
                                          "newObjectId": "0" * 40}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"), f"couldn't delete {branch}", branch)


def ado_default_head(gp):
    """(default branch name, its head commit) of the repository."""
    repo = ado(gp, "GET", "")
    name = (repo.get("defaultBranch") or "refs/heads/main")[len("refs/heads/"):]
    ref = ado_ref(gp, name)
    return name, (ref or {}).get("objectId")


def ado_tree(gp, commit, missing_ok=False):
    """Every file path under the connection's Git folder at a commit ([] if the folder is missing and missing_ok)."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    try:
        body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                                "versionDescriptor.version": commit,
                                                "versionDescriptor.versionType": "commit"})
    except HttpError as exc:
        if missing_ok and exc.status == 404:
            return []
        raise
    return [e["path"] for e in body.get("value", []) if e.get("gitObjectType") == "blob"]


def ado_file(gp, commit, path):
    return ado(gp, "GET", "/items", params={"path": path, "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit", "$format": "octetStream"},
               raw=True)


def git_items_at(gp, commit):
    """Fabric items saved in Git at a commit: {key_of(type, name): {type, displayName, logicalId, folder, files}}."""
    paths = ado_tree(gp, commit)
    out = {}
    for p in paths:
        if not p.endswith("/.platform"):
            continue
        folder = p[: -len("/.platform")]
        try:
            meta = json.loads(ado_file(gp, commit, p).decode("utf-8-sig"))
        except ValueError as exc:
            raise HttpError(None, "BadPlatformFile", f"{p} is not valid JSON: {exc}", p) from None
        md, cfg = meta.get("metadata") or {}, meta.get("config") or {}
        files = [q for q in paths if q.startswith(folder + "/")]
        out[key_of(md.get("type"), md.get("displayName"))] = {
            "type": md.get("type"), "displayName": md.get("displayName"), "logicalId": cfg.get("logicalId"),
            "folder": folder, "files": len(files)}
    return out


# ---------------------------------------------------------------- Fabric Git integration
FOLDER_MARKER = "README.md"     # placeholder V2_1 writes so the workspace's Git folder exists before linking


def folder_marker_path(folder):
    return "/" + folder.strip("/") + "/" + FOLDER_MARKER


def ado_create_folder(gp, branch, folder, text):
    """Create a Git folder by pushing a small marker file to the branch: the Fabric Git connect API (unlike the
    workspace settings screen) doesn't create a missing folder and fails with GitProviderResourceNotFound.
    Returns the new branch head."""
    ref = ado_ref(gp, branch)
    if not ref:
        raise HttpError(None, "BranchNotFound", f"branch {branch} doesn't exist", branch)
    res = ado(gp, "POST", "/pushes", body={
        "refUpdates": [{"name": f"refs/heads/{branch}", "oldObjectId": ref["objectId"]}],
        "commits": [{"comment": f"Create Git folder {folder} for Fabric Git integration",
                     "changes": [{"changeType": "add", "item": {"path": folder_marker_path(folder)},
                                  "newContent": {"content": text, "contentType": "rawtext"}}]}]})
    return ((res.get("commits") or [{}])[0]).get("commitId")


def ado_tree_ids(gp, commit):
    """{path: blob objectId} under the connection's Git folder at a commit: compares folder content cheaply."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                            "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit"})
    return {e["path"]: e.get("objectId") for e in body.get("value", []) if e.get("gitObjectType") == "blob"}


def ado_create_tag(gp, name, commit, message):
    """Annotated tag (refs/tags/<name>) on a commit. Returns the tag's commit, or raises."""
    existing = ado(gp, "GET", "/refs", params={"filter": f"tags/{name}"}).get("value", [])
    if any(r.get("name") == f"refs/tags/{name}" for r in existing):
        raise HttpError(None, "TagExists", f"tag {name} already exists", name)
    return ado(gp, "POST", "/annotatedtags", body={"name": name, "taggedObject": {"objectId": commit},
                                                    "message": message})


# ---------------------------------------------------------------- who runs this, and their workspace role
def token_claims(audience="pbi"):
    """Claims of the runner's Entra token (no secret is printed): oid, upn, idtyp ('app' = service principal)."""
    try:
        part = notebookutils.credentials.getToken(audience).split(".")[1]
        return json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
    except Exception:
        return {}


def me():
    c = token_claims()
    return {"oid": str(c.get("oid", "")).lower(), "upn": c.get("upn") or c.get("unique_name") or c.get("appid", ""),
            "tid": str(c.get("tid", "")).lower(),
            "isApp": c.get("idtyp") == "app" or ("appid" in c and "upn" not in c and "unique_name" not in c)}


def _tenant_from_headers(headers):
    tid = headers.get("X-VSS-ResourceTenant") or headers.get("x-vss-resourcetenant")
    if tid is None:
        m = re.search(r"login\.(?:microsoftonline\.com|windows\.net)/([0-9a-fA-F-]{36})",
                      headers.get("WWW-Authenticate") or headers.get("www-authenticate") or "")
        tid = m.group(1) if m else None
    if tid is None:
        return None
    tid = tid.split(",")[0].strip().lower()
    return "" if tid in ("", "00000000-0000-0000-0000-000000000000") else tid


def sp_settings_missing():
    return [n for n in ("ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME")
            if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def sp_repo_check(org, project, repo):
    """(True, '') if the service principal in ADO_SP_* can read the repository, else (False, error)."""
    url = f"https://dev.azure.com/{quote(org)}/{quote(project)}/_apis/git/repositories/{quote(repo)}"
    try:
        ado_call(url, auth=sp_header())
        return True, ""
    except HttpError as exc:
        return False, exc.short()


def ado_org_tenant_info(org, project=None, repo=None):
    """(tenant ID, how it was found) of the Entra tenant the DevOps organization is connected to. '' = not connected
    to Entra, None = couldn't tell. 1) Azure DevOps names it in the X-VSS-ResourceTenant (or WWW-Authenticate) header
    of an anonymous call. 2) Otherwise, if the service principal in ADO_SP_* can read the repository, the organization
    is connected to the service principal's tenant: Azure DevOps only accepts service principals of its own tenant."""
    for path in ("_apis/projects?api-version=" + ADO_API_VERSION, "", "_apis/connectionData"):
        try:
            resp = _session().get(f"https://dev.azure.com/{quote(org)}/{path}", timeout=30, allow_redirects=False)
        except requests.RequestException:
            continue
        tid = _tenant_from_headers(resp.headers)
        if tid is not None:
            return tid, "named by Azure DevOps"
    if project and repo and not sp_settings_missing():
        try:
            ok, _ = sp_repo_check(org, project, repo)
        except HttpError:
            ok = False
        if ok:
            return (str(ADO_SP_TENANT_ID).strip().lower(),
                    f"the organization accepts service principal {str(ADO_SP_CLIENT_ID).strip()} of tenant "
                    f"{str(ADO_SP_TENANT_ID).strip()}, and Azure DevOps only accepts service principals of the Entra "
                    "tenant it's connected to")
    return None, ("Azure DevOps didn't name its tenant, and the service principal couldn't confirm it" if project
                  else "Azure DevOps didn't name its tenant")


def ado_org_tenant(org, project=None, repo=None):
    return ado_org_tenant_info(org, project, repo)[0]


def my_workspace_role(ws_id):
    """Your role in a workspace via the admin API (Fabric administrator): 'Admin', 'Member', ..., '' = none,
    None = couldn't tell."""
    who = me()
    try:
        users = fabric("GET", f"/admin/workspaces/{ws_id}/users").get("accessDetails") or []
    except HttpError:
        return None
    for u in users:
        pr = u.get("principal") or {}
        upn = ((pr.get("userDetails") or {}).get("userPrincipalName") or "").lower()
        if str(pr.get("id", "")).lower() == who["oid"] or (upn and upn == str(who["upn"]).lower()):
            return (u.get("workspaceAccessDetails") or {}).get("workspaceRole") or ""
    return ""


def add_me_as_workspace_admin(ws_id):
    """Fabric/Power BI administrator API: add the runner as Admin of a workspace."""
    who = me()
    http("POST", f"https://api.powerbi.com/v1.0/myorg/admin/groups/{ws_id}/users", "pbi",
         json_body={"identifier": who["oid"], "principalType": "App" if who["isApp"] else "User",
                    "groupUserAccessRight": "Admin"}, label="fabric")


CROSS_GEO_TITLE = "Users can export items to Git repositories in other geographical locations"
CROSS_GEO_FIX = ("The workspace's capacity is in another geography than the Azure DevOps organization. A Fabric "
                 "administrator must enable Admin portal > Tenant settings > Git integration > '" + CROSS_GEO_TITLE +
                 "' for the entire organization, or for a security group that includes everyone who runs V2_1 to "
                 "V2_5 (the setting applies to the person doing the Git operation, not to the DevOps account). Only "
                 "item definitions cross geographies, never data. Wait a few minutes after Apply, then rerun: the "
                 "notebook resumes. Linking by hand fails the same way until the setting is on.")


def git_tenant_settings():
    """Git integration tenant settings from the admin API, by title (admin role needed)."""
    res = fabric("GET", "/admin/tenantsettings")
    return {t.get("title", t.get("settingName", "")): t for t in res.get("tenantSettings", [])
            if "git" in (t.get("title", "") + t.get("tenantSettingGroup", "")).lower()}


def git_connection_url(org, project, repo):
    return f"https://dev.azure.com/{quote(org)}/{quote(project)}/_git/{quote(repo)}/"


def find_git_connection(name):
    """Fabric connection with this display name that you can use, or None."""
    return next((c for c in fabric_pages("/connections") if c.get("displayName") == name), None)


def create_git_connection(name, org, project, repo):
    """Fabric cloud connection 'Azure DevOps - Source control' with the service principal in ADO_SP_*. The
    client secret is read from Key Vault and only sent to Fabric."""
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    body = {"displayName": name, "connectivityType": "ShareableCloud",
            "connectionDetails": {"type": "AzureDevOpsSourceControl", "creationMethod": "AzureDevOpsSourceControl.Contents",
                                  "parameters": [{"dataType": "Text", "name": "url",
                                                  "value": git_connection_url(org, project, repo)}]},
            "credentialDetails": {"credentials": {"credentialType": "ServicePrincipal",
                                                  "tenantId": str(ADO_SP_TENANT_ID).strip(),
                                                  "servicePrincipalClientId": str(ADO_SP_CLIENT_ID).strip(),
                                                  "servicePrincipalSecret": secret}}}
    return fabric("POST", "/connections", body)


GIT_LINK_MODES = {"service_principal": "service principal (Fabric connection)",
                  "my_account": "your own Azure DevOps sign-in",
                  "connection_id": "an existing Fabric connection (GIT_CONNECTION_ID)"}


def git_link_mode():
    """How Fabric signs in to Azure DevOps for the workspaces' Git link (GIT_LINK_AUTH)."""
    raw = globals().get("GIT_LINK_AUTH")
    if raw is None:                       # older settings cells
        if globals().get("CREATE_GIT_CONNECTION"):
            return "service_principal"
        return "connection_id" if str(globals().get("GIT_CONNECTION_ID") or "").strip() else "my_account"
    m = str(raw).strip().lower().replace("-", "_").replace(" ", "_")
    m = {"sp": "service_principal", "spn": "service_principal", "serviceprincipal": "service_principal",
         "user": "my_account", "me": "my_account", "automatic": "my_account", "sso": "my_account",
         "connection": "connection_id", "configuredconnection": "connection_id"}.get(m, m)
    if m not in GIT_LINK_MODES:
        raise ValueError(f"GIT_LINK_AUTH = {raw!r}: use \"service_principal\", \"my_account\" or \"connection_id\"")
    return m


def _read_git_connection(cid):
    try:
        c = fabric("GET", f"/connections/{cid}")
    except HttpError as exc:
        raise RuntimeError(f"Can't read Fabric connection {cid} ({exc.short()}): check GIT_CONNECTION_ID, and that you "
                           "have User permission on it (Settings > Manage connections and gateways).") from None
    t = (c.get("connectionDetails") or {}).get("type")
    if t != "AzureDevOpsSourceControl":
        raise RuntimeError(f"Fabric connection {cid} ({c.get('displayName')}) is of type {t}, not Azure DevOps - "
                           "Source control: choose another connection.")
    return c


def plan_git_link(who, org, project, repo, allow_create=True):
    """How Fabric will sign in to Azure DevOps when it links a workspace, from GIT_LINK_AUTH. Returns {mode,
    connectionId, create (name of the connection to create on confirm, or None), text}. Raises before anything
    changes when the choice can't work."""
    mode = git_link_mode()
    cid = str(globals().get("GIT_CONNECTION_ID") or "").strip()
    if mode == "my_account":
        if cid:
            raise ValueError("GIT_CONNECTION_ID is filled but GIT_LINK_AUTH = \"my_account\": empty GIT_CONNECTION_ID, "
                             "or set GIT_LINK_AUTH = \"connection_id\" to use that connection.")
        if who["isApp"]:
            raise RuntimeError("This notebook runs as a service principal / workspace identity, which has no Azure "
                               "DevOps sign-in of its own: set GIT_LINK_AUTH = \"service_principal\".")
        return {"mode": mode, "connectionId": "", "create": None,
                "text": f"your own Azure DevOps sign-in ({who['upn']}). That only works if the DevOps organization "
                        f"is connected to the same Entra tenant as Fabric and {who['upn']} is a member of it, with "
                        "Basic access and project Contributors."}
    if mode == "connection_id":
        if not cid:
            raise ValueError("GIT_LINK_AUTH = \"connection_id\" needs GIT_CONNECTION_ID.")
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"the identity stored in Fabric connection {c.get('displayName')} ({cid}). Give that identity "
                        "Basic access and project Contributors in Azure DevOps."}
    need = sp_settings_missing()
    if need:
        raise ValueError("GIT_LINK_AUTH = \"service_principal\" needs: " + ", ".join(need) +
                         " (fill them even if ADO_AUTH = \"pat\").")
    sp_tid = str(ADO_SP_TENANT_ID).strip().lower()
    if who.get("tid") and sp_tid != who["tid"] and not globals().get("ALLOW_CROSS_TENANT"):
        raise RuntimeError(f"Stopped: the service principal's tenant ({sp_tid}) isn't the Fabric tenant ({who['tid']}). "
                           "Use an app registration from the same tenant (ALLOW_CROSS_TENANT is not part of this "
                           "migration).")
    sp_text = (f"service principal {str(ADO_SP_CLIENT_ID).strip()} (not tied to a person). Fabric stores its client "
               "secret in the connection: when you renew the secret, update the connection too.")
    if cid:
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"Fabric connection {c.get('displayName')} ({cid}), which signs in as the " + sp_text}
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    found = find_git_connection(name)
    if found:
        t = (found.get("connectionDetails") or {}).get("type")
        cred = (found.get("credentialDetails") or {}).get("credentialType")
        if t != "AzureDevOpsSourceControl" or (cred and cred != "ServicePrincipal"):
            raise RuntimeError(f"A Fabric connection named {name} already exists but it's {t} / {cred}, not Azure DevOps "
                               "- Source control with a service principal: set another GIT_CONNECTION_NAME.")
        return {"mode": mode, "connectionId": found["id"], "create": None,
                "text": f"existing Fabric connection {name} ({found['id']}), which signs in as the " + sp_text}
    if not allow_create:
        return {"mode": mode, "connectionId": "", "create": name,
                "text": f"Fabric connection {name} (doesn't exist yet), which will sign in as the " + sp_text}
    return {"mode": mode, "connectionId": "", "create": name,
            "text": f"Fabric connection {name}, created when you confirm the plan, which signs in as the " + sp_text}


SAME_TENANT_FIX = (
    "This migration keeps everything in one Entra tenant: the Azure DevOps organization must be connected to the "
    "same Entra tenant as Fabric. Either use a DevOps organization that is already in this tenant, or connect this "
    "one: Azure DevOps > Organization settings > Microsoft Entra > Connect directory (Project Collection "
    "Administrator; existing users must exist in the tenant or they lose access, so plan it with the DevOps admin). "
    "Then add the people who run the notebooks (Basic access, project Contributors), open a workspace > Source "
    "control once and sign in. Linking across tenants (ALLOW_CROSS_TENANT = True) is not part of this migration.")
FOLDER_FIRST = (
    "Most common cause: the Git folder doesn't exist in the branch (the connect API doesn't create it, unlike the "
    "workspace settings screen). V2_1 creates it before linking; if you link by hand or with another tool, create "
    "the folder first (a file inside it, for example README.md). ")
GIT_NOT_FOUND_FIX = (
    "Fabric couldn't see the organization, project, repository or branch with your own Azure DevOps sign-in. This "
    "notebook already found all of them with the PAT/service principal, so the names are right (those settings are "
    "only for the notebook's own DevOps calls). With your own sign-in Fabric only sees DevOps organizations connected "
    "to this Fabric tenant's Entra ID in which you are a member. Check, in this order: (1) you are a member of the "
    "organization with Basic access and in the project's Contributors; (2) the organization is connected to this "
    "Entra tenant (V2_0 row 'organization in the same Entra tenant as Fabric'). " + SAME_TENANT_FIX + " Optional, "
    "still within the same tenant: GIT_LINK_AUTH = \"service_principal\" links with the service principal "
    "instead of your own account.")
GIT_NOT_FOUND_WITH_CONNECTION_FIX = (
    "Fabric couldn't see the repository, branch or folder with the Git connection's identity. " + FOLDER_FIRST +
    "Otherwise: Fabric couldn't see the repository with the Git connection's identity. Give that identity (for example the "
    "service principal) access: Azure DevOps > Organization settings > Users > Add (Basic), and the project's "
    "Contributors group. Check that the connection URL points to this organization/project/repository, and that "
    "the connection's tenant ID is the tenant the DevOps organization is connected to.")


def git_connect_fix(exc):
    """How to fix a failed Git connect / initialize call."""
    text = f"{getattr(exc, 'code', '')} {exc}".lower()
    if "principaltypenotsupported" in text or "service principal" in text:
        return ("The notebook runs as a service principal / workspace identity: Fabric only accepts that with a "
                "stored connection: set GIT_LINK_AUTH = \"service_principal\", or run the notebook interactively as "
                "a user.")
    if "gitproviderresourcenotfound" in text:
        return FOLDER_FIRST + "Otherwise: " + GIT_NOT_FOUND_FIX if git_link_mode() == "my_account" else \
            GIT_NOT_FOUND_WITH_CONNECTION_FIX
    if "insufficientprivileges" in text or getattr(exc, "status", None) == 403:
        return ("You must be Admin of the workspace (Fabric administrator isn't enough). Set "
                "ADD_ME_AS_WORKSPACE_ADMIN = True, or ask a workspace admin to add you.")
    if "credential" in text or "gitprovider" in text or "tf400813" in text or "unauthorized" in text:
        return ("Fabric has no working Azure DevOps credentials for you. Either open any workspace > Source control "
                "once and sign in to Azure DevOps (same Entra tenant as the DevOps organization), or set "
                "GIT_LINK_AUTH = \"service_principal\" so Fabric signs in with the service principal instead.")
    if "crossgeo" in text or "cross geo" in text or "cross-geo" in text:
        return CROSS_GEO_FIX
    if "capacity" in text:
        return "Assign the workspace to a Fabric capacity first."
    if "alreadyconnected" in text:
        return "Already connected: rerun the notebook, it will check the existing link."
    if "directory" in text or "folder" in text or "branch" in text:
        return "Check the branch and folder: the folder must be empty or belong to this workspace."
    return "See the error; if it persists, link this workspace by hand with the values in the table."


def current_workspace_id():
    """Workspace this notebook runs in ('' if unknown)."""
    try:
        ctx = notebookutils.runtime.context
        return str(ctx.get("currentWorkspaceId") or "").lower()
    except Exception:
        return ""


RUN_FROM_ELSEWHERE = ("Run the migration notebooks from a separate workspace that isn't migrated (for example "
                      "'fabric-migration-control'): this workspace is in scope, so its notebooks, including the one "
                      "running, would be deleted with the other items.")


def my_git_credentials(ws_id):
    """Your own Git credentials for a linked workspace ({'source': 'Automatic' | 'ConfiguredConnection' | 'None',
    'connectionId'}). Per user and per workspace: the workspace screens (Source control, Git integration) use them."""
    try:
        return fabric("GET", f"/workspaces/{ws_id}/git/myGitCredentials")
    except HttpError as exc:
        return {"source": "?", "error": exc.short()}


def creds_need_fix(cred, connection_id):
    """True if your Git credentials for the workspace should be switched to the Fabric connection."""
    if git_link_mode() == "my_account":
        return False
    return cred.get("source") != "ConfiguredConnection" or bool(connection_id and cred.get("connectionId") != connection_id)


def set_my_git_credentials(ws_id, connection_id):
    fabric("PATCH", f"/workspaces/{ws_id}/git/myGitCredentials",
           {"source": "ConfiguredConnection", "connectionId": connection_id})


CREDS_WHY = ("your own Git sign-in for this workspace is '{src}', so the workspace screens (Source control, Git "
             "integration) try your personal Azure DevOps account and show 'Can't fetch branches' / 'connection "
             "removed'. The link itself is fine.")


def git_connection_label():
    """Name of the Fabric connection used to link (GIT_CONNECTION_NAME or the default name)."""
    if git_link_mode() == "connection_id":
        return f"with ID {globals().get('GIT_CONNECTION_ID')}"
    return (str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{globals().get('ADO_ORGANIZATION')}"
            f"-{globals().get('ADO_PROJECT')}-{globals().get('ADO_REPOSITORY')}")


def git_creds_hint(ws_id, gp=None):
    """Plain-language fix for a Git call (status, commit, update, disconnect) that fails because of YOUR OWN Git
    sign-in for the workspace. The notebooks don't change that sign-in by themselves: you pick it in the workspace."""
    cred = my_git_credentials(ws_id)
    gp = gp or {}
    org, project, repo = (gp.get("organizationName") or globals().get("ADO_ORGANIZATION", ""),
                          gp.get("projectName") or globals().get("ADO_PROJECT", "<project>"),
                          gp.get("repositoryName") or globals().get("ADO_REPOSITORY", "<repository>"))
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    src = cred.get("source")
    now = f"Your Git sign-in for this workspace is '{src}'" + (
        f" (connection {cred.get('connectionId')})" if cred.get("connectionId") else "") + "."
    return (f"{now} Git calls run with YOUR OWN Git sign-in for each workspace. Open the workspace > Source control "
            f"> account/credentials and choose the Fabric connection '{name}' (the one used to link it; if it isn't "
            "listed, ask its owner to share it with you: Manage connections and gateways > Manage users), then rerun. "
            "If the workspace was linked with your own account (GIT_LINK_AUTH = \"my_account\"), sign in to Azure "
            "DevOps there instead.")


def git_fail_text(exc, ws_id, gp=None):
    """Full error text for a failed Fabric Git call plus the fix: role (403) or your own Git sign-in (other errors)."""
    text = exc.short()
    if "/git/" not in str(getattr(exc, "url", "") or ""):
        return text
    if exc.status == 403:
        return (text + " | FIX: Git calls need Contributor (status, commit, update) or Admin (connect, disconnect) on "
                "the workspace for the account running the notebook.")
    try:
        return text + " | FIX: " + git_creds_hint(ws_id, gp)
    except Exception:          # the hint is best effort; never hide the original error
        return text


def git_connection(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/connection")


def git_status(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/status")


def resolve_target_workspace(old_id, name, override=None, recorded=None):
    """(workspace ID after the move, how it was found) or (None, why not). A home-region move keeps the workspace
    IDs, so the same ID is tried first; if it's gone, the workspace with the same name (names are unique in a
    tenant). override = WORKSPACE_MAP entry; recorded = the target V2_5 already used (restore_state.json)."""
    if override:
        return override, "WORKSPACE_MAP"
    if recorded:
        return recorded, "same as V2_5" + ("" if recorded == old_id else " (ID changed)")
    try:
        fabric("GET", f"/workspaces/{old_id}")
        return old_id, "same ID"
    except HttpError as exc:
        if exc.status not in (403, 404):
            raise
    try:
        found = [w for w in fabric_pages("/workspaces") if (w.get("displayName") or "") == name]
    except HttpError as exc:
        return None, f"workspace {old_id} not found and the workspace list can't be read ({exc.short()})"
    if len(found) == 1:
        return norm_id(found[0]["id"]), f"found by name; ID changed from {old_id}"
    if not found:
        return None, (f"workspace {old_id} ({name}) not found, and no workspace you can access has that name. "
                      "Recreate it or add yourself to it; if it was renamed, set WORKSPACE_MAP")
    return None, f"workspace {old_id} not found and {len(found)} workspaces are named {name}: set WORKSPACE_MAP"


def workspace_items(ws_id):
    return [{"id": i["id"], "type": i["type"], "displayName": i["displayName"], "workspaceId": ws_id}
            for i in fabric_pages(f"/workspaces/{ws_id}/items")]


# ---------------------------------------------------------------- migration records (ADLS)
def migration_prefix(snapshot_id):
    return f"{ADLS_ROOT.strip('/')}/_migration/{snapshot_id}"


def plan_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


def load_manifest(snapshot_id):
    """The manifest written by the snapshot notebook, checked against its stored hash."""
    if not snapshot_id or "<" in snapshot_id:
        raise ValueError("Set SNAPSHOT_ID to the snapshot ID printed by the snapshot notebook.")
    prefix = migration_prefix(snapshot_id)
    try:
        raw = adls_read(f"{prefix}/manifest.json")
        stored = adls_read(f"{prefix}/manifest.sha256").decode().strip()
    except HttpError as exc:
        raise RuntimeError(f"Can't read the snapshot manifest under {prefix}/ ({exc.short()}).") from None
    if hashlib.sha256(raw).hexdigest() != stored:
        raise RuntimeError("The manifest doesn't match its stored hash: it was changed after the snapshot. Stop and "
                           "investigate; take a new snapshot.")
    return json.loads(raw)


def save_log(prefix, name):
    df = pd.DataFrame(RESULTS, columns=["Workspace", "Item", "Type", "ItemId", "Step", "Status", "Detail"])
    adls_write(f"{prefix}/{name}_{NOW_ID}.csv", df.to_csv(index=False))
    return df


# ---------------------------------------------------------------- data restore helpers (V2_6 only)
KQL_TIMEOUT_SECONDS = 4 * 60 * 60
KQL_TERMINAL = {"Completed", "Failed", "PartiallySucceeded", "Abandoned", "BadInput", "Canceled", "Cancelled",
                "Throttled"}


def adls_read_range(path, start, end):
    return http("GET", adls_url(path), "storage", headers={**ADLS_HEADERS, "Range": f"bytes={start}-{end}"},
                label="adls").content


def onelake_url(workspace_id, item_id, rel_path):
    return f"{ONELAKE_DFS}/{workspace_id}/{item_id}/{quote(rel_path, safe='/')}"


def onelake_list(workspace_id, item_id, folder):
    """Every file under <item>/<folder> as (relative path, size). Empty if the folder is missing."""
    params = {"resource": "filesystem", "recursive": "true", "directory": f"{item_id}/{folder}"}
    out = []
    while True:
        try:
            resp = http("GET", f"{ONELAKE_DFS}/{workspace_id}", "storage", headers=ADLS_HEADERS, params=params,
                        label="onelake")
        except HttpError as exc:
            if exc.status == 404:
                return out
            raise
        for e in resp.json().get("paths", []):
            if str(e.get("isDirectory", "false")).lower() == "true":
                continue
            rel = e["name"].split("/", 1)[1] if "/" in e["name"] else e["name"]
            out.append((rel, int(e.get("contentLength") or 0)))
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return out
        params = {**params, "continuation": token}


def onelake_size(workspace_id, item_id, rel_path):
    """Size of a OneLake file, or None if it doesn't exist."""
    try:
        resp = http("HEAD", onelake_url(workspace_id, item_id, rel_path), "storage", headers=ADLS_HEADERS,
                    label="onelake")
        return int(resp.headers.get("Content-Length") or 0)
    except HttpError as exc:
        if exc.status == 404:
            return None
        raise


def onelake_delete(workspace_id, item_id, rel_path):
    http("DELETE", onelake_url(workspace_id, item_id, rel_path), "storage", headers=ADLS_HEADERS, label="onelake")


def restore_file(src_path, size, workspace_id, item_id, rel_path, may_replace_partial):
    """Copy one backup file into OneLake. Never overwrites a complete file: returns "copied" or "already there".
    may_replace_partial: this notebook started loading the item, so a different-size file is a leftover of an
    interrupted run and may be replaced."""
    url = onelake_url(workspace_id, item_id, rel_path)
    try:
        http("PUT", url, "storage", headers={**ADLS_HEADERS, "If-None-Match": "*"}, params={"resource": "file"},
             label="onelake", retry=False)
    except HttpError as exc:
        if exc.status != 409:
            raise
        existing = onelake_size(workspace_id, item_id, rel_path)
        if existing == size:
            return "already there"
        if not may_replace_partial:
            raise HttpError(409, "Exists", f"a different file already exists ({existing} bytes, backup {size})", url)
        http("PUT", url, "storage", headers=ADLS_HEADERS, params={"resource": "file"}, label="onelake")
    pos = 0
    while pos < size:
        chunk = adls_read_range(src_path, pos, min(pos + CHUNK_BYTES, size) - 1)
        if not chunk:
            raise HttpError(None, "ShortRead", f"no bytes at offset {pos}", src_path)
        http("PATCH", url, "storage", headers=ADLS_HEADERS, data=chunk,
             params={"action": "append", "position": str(pos)}, label="onelake")
        pos += len(chunk)
    http("PATCH", url, "storage", headers=ADLS_HEADERS, data=b"", params={"action": "flush", "position": str(pos)},
         label="onelake")
    return "copied"


def abfss_adls(path):
    return f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net/{path.strip('/')}"


def abfss_onelake(workspace_id, item_id, rel_path=""):
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{item_id}" + (f"/{rel_path.strip('/')}" if rel_path.strip("/") else "")


def onelake_mkdir(workspace_id, item_id, rel_path):
    """Create a OneLake folder (and its parents); no error if it exists."""
    http("PUT", onelake_url(workspace_id, item_id, rel_path), "storage", headers=ADLS_HEADERS,
         params={"resource": "directory"}, label="onelake")


FASTCP_FLAGS = "--overwrite=false"            # never replace a file that is already there
FASTCP_LAYOUT = None                          # "into" / "contents" once probed; "" = fastcp not usable here


def fastcp(src, dst, exclude_delta_log=False):
    flags = FASTCP_FLAGS + (" --exclude-path=_delta_log" if exclude_delta_log else "")
    ok = notebookutils.fs.fastcp(src, dst, True, {"flags": flags})
    if ok is False:
        raise RuntimeError(f"fastcp returned False for {src}")


def fastcp_layout():
    """How notebookutils.fs.fastcp places a folder, learned once on a tiny ADLS-to-ADLS copy (nothing is written to
    any Fabric item): "into" = copying folder T to an existing folder P gives P/T (azcopy), "contents" = it gives P
    itself. Also checks that the _delta_log exclusion works. Returns (layout or "", reason)."""
    global FASTCP_LAYOUT
    if FASTCP_LAYOUT is not None:
        return FASTCP_LAYOUT, ""
    base = f"{PREFIX}/_fastcp_probe_{NOW_ID}"
    reason = ""
    try:
        adls_write(f"{base}/src/t/keep.bin", b"k")
        adls_write(f"{base}/src/t/_delta_log/skip.json", b"s")
        adls_write(f"{base}/dst/_marker", b"m")
        fastcp(abfss_adls(f"{base}/src/t"), abfss_adls(f"{base}/dst"), exclude_delta_log=True)
        got = {p for p, _, d in adls_list(f"{base}/dst") if not d} - {"_marker"}
        if got == {"t/keep.bin"}:
            FASTCP_LAYOUT = "into"
        elif got == {"keep.bin"}:
            FASTCP_LAYOUT = "contents"
        else:
            reason = (f"unexpected layout {sorted(got)}" if not any("_delta_log" in p for p in got)
                      else "the _delta_log exclusion isn't applied")
    except Exception as exc:
        reason = (exc.short() if isinstance(exc, HttpError) else str(exc))[:300]
    finally:
        try:
            http("DELETE", adls_url(base), "storage", headers=ADLS_HEADERS, params={"recursive": "true"}, label="adls")
        except Exception:
            pass
    if FASTCP_LAYOUT is None:
        FASTCP_LAYOUT = ""
    return FASTCP_LAYOUT, reason


def table_root(rel):
    """'Tables/dbo/sales/_delta_log/0001.json' -> 'Tables/dbo/sales'; None outside a Delta table."""
    return rel.split("/_delta_log/")[0] if "/_delta_log/" in rel else None


def kusto_mgmt(query_uri, database, command, retry=True, kind="mgmt"):
    resp = http("POST", query_uri.rstrip("/") + f"/v1/rest/{kind}", "kusto",
                json_body={"db": database, "csl": command}, label="kusto", retry=retry)
    tables = resp.json().get("Tables") or []
    if not tables:
        return []
    cols = [c["ColumnName"] for c in tables[0].get("Columns", [])]
    return [dict(zip(cols, row)) for row in tables[0].get("Rows", [])]


def kql_string(text):
    return "'" + str(text).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kusto_wait(uri, db, op, label):
    deadline = time.monotonic() + KQL_TIMEOUT_SECONDS
    while True:
        time.sleep(5)
        state = kusto_mgmt(uri, db, f".show operations {op}")[0]
        if state.get("State") in KQL_TERMINAL:
            if state.get("State") != "Completed":
                raise HttpError(None, state.get("State"), state.get("Status") or label, uri)
            return
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", f"{label}: operation {op} still running", uri)


def kusto_count(uri, db, table):
    rows = kusto_mgmt(uri, db, f"[{kql_string(table)}] | count", kind="query")
    return int(rows[0].get("Count", 0)) if rows else 0


def delta_mapped_columns(metadata):
    """Validated logical/physical names and SQL types for a flat, column-mapped Warehouse table."""
    try:
        schema = json.loads(metadata.get("schemaString") or "")
    except (TypeError, ValueError) as exc:
        return [], [f"column mapping has no readable schemaString: {exc}"]
    if not isinstance(schema, dict) or schema.get("type") != "struct" or not schema.get("fields"):
        return [], ["column mapping schemaString must describe a non-empty struct"]
    if metadata.get("partitionColumns"):
        return [], ["partitioned column-mapped tables need loading by hand: partition values aren't in Parquet"]
    sql_types = {"boolean": "bit", "byte": "smallint", "short": "smallint", "integer": "int", "long": "bigint",
                 "float": "real", "double": "float", "string": "varchar(max)", "binary": "varbinary(max)",
                 "date": "date", "timestamp": "datetime2(6)", "timestamp_ntz": "datetime2(6)"}
    columns, logical, physical = [], set(), set()
    for field in schema["fields"]:
        if not isinstance(field, dict):
            return [], ["column mapping schema contains an invalid field"]
        name, typ = field.get("name"), field.get("type")
        props = field.get("metadata") or {}
        source = props.get("delta.columnMapping.physicalName") if isinstance(props, dict) else None
        if not all(isinstance(n, str) and n and len(n) <= 128 and "\0" not in n for n in (name, source)):
            return [], [f"column mapping field {name!r} has no valid logical/physical name"]
        if "." in source:
            return [], [f"column mapping physical name {source!r} can't be bound as a flat Parquet column"]
        if name in logical or source in physical:
            return [], [f"column mapping has duplicate logical/physical names for {name!r}"]
        sql_type = sql_types.get(typ) if isinstance(typ, str) else None
        decimal = re.fullmatch(r"decimal\((\d+),\s*(\d+)\)", typ) if isinstance(typ, str) else None
        if decimal and 1 <= int(decimal[1]) <= 38 and 0 <= int(decimal[2]) <= int(decimal[1]):
            sql_type = f"decimal({int(decimal[1])},{int(decimal[2])})"
        if not sql_type:
            return [], [f"column mapping field {name!r} has unsupported Delta type {typ!r}"]
        logical.add(name)
        physical.add(source)
        columns.append({"name": name, "physical": source, "type": sql_type})
    return columns, []


def delta_active_files(folder, table, files):
    """Replay the Delta snapshot, retaining mapped schema and known null counts for verified Warehouse loads."""
    logs = sorted(rel for rel, _ in files if rel.startswith(table + "/_delta_log/")
                  and re.search(r"/\d{20}\.json$", rel))
    active, problems, metadata = {}, [], None
    start = 0
    versions = [int(rel[-25:-5]) for rel in logs]
    if not versions or versions[0] != 0:
        cp = next((rel for rel, _ in files if rel == table + "/_delta_log/_last_checkpoint"), None)
        if not cp:
            return None, ["the Delta log has no version 0 and no checkpoint"]
        try:
            import pyarrow.parquet as pq, io as _io
            ver = int(json.loads(adls_read(f"{folder}/data/{cp}"))["version"])
            start = ver + 1
            checkpoints = [rel for rel, _ in files if rel.startswith(table + "/_delta_log/") and
                           re.search(rf"/{ver:020d}\.checkpoint(\.\d+\.\d+)?\.parquet$", rel)]
            if not checkpoints:
                return None, [f"Delta checkpoint {ver} is missing from the backup"]
            for rel in checkpoints:
                for row in pq.read_table(_io.BytesIO(adls_read(f"{folder}/data/{rel}"))).to_pylist():
                    if row.get("add"):
                        active[row["add"]["path"]] = row["add"]
                    if row.get("metaData"):
                        metadata = row["metaData"]
        except Exception as exc:
            return None, [f"couldn't read the Delta checkpoint: {exc}"]
    tail = [v for v in versions if v >= start]
    if tail and (tail[0] != start or any(b != a + 1 for a, b in zip(tail, tail[1:]))):
        return None, ["the backed-up Delta log has missing versions"]
    for rel in logs:
        if int(rel[-25:-5]) < start:
            continue
        for line in adls_read(f"{folder}/data/{rel}").decode("utf-8").splitlines():
            if not line.strip():
                continue
            action = json.loads(line)
            if "add" in action:
                active[action["add"]["path"]] = action["add"]
            elif "remove" in action:
                active.pop(action["remove"]["path"], None)
            elif "metaData" in action:
                metadata = action["metaData"]
    if metadata is None:
        return None, ["the Delta snapshot has no schema metadata"]
    try:
        mode = dict(metadata.get("configuration") or {}).get("delta.columnMapping.mode", "none")
    except (TypeError, ValueError):
        return None, ["the Delta snapshot has invalid schema configuration"]
    columns = []
    if mode in ("name", "id"):
        columns, mapping_problems = delta_mapped_columns(metadata)
        problems += mapping_problems
    elif mode != "none":
        problems.append(f"unsupported Delta column mapping mode {mode!r}")
    if any(a.get("deletionVector") for a in active.values()):
        problems.append("deletion vectors: some files have deleted rows that COPY INTO would bring back")
    have = {rel for rel, _ in files}
    missing = [p for p in active if f"{table}/{p}" not in have]
    if missing:
        problems.append("active Delta data files are missing from the backup: " + ", ".join(missing))
    rows, statistics = 0, []
    for a in active.values():
        try:
            stats = json.loads(a.get("stats") or "{}")
            n = stats["numRecords"]
            if not isinstance(n, int) or isinstance(n, bool) or n < 0:
                raise ValueError("invalid row count")
            rows += n
            statistics.append(stats)
        except (TypeError, ValueError, KeyError):
            rows = None
            break
    result = {"files": sorted(active), "rows": rows}
    if columns:
        result["columns"], result["nullCounts"] = columns, {}
        if rows is None:
            problems.append("column-mapped table has no complete row count in its Delta statistics")
        else:
            nulls = [s.get("nullCount") for s in statistics]
            if any(n is not None and not isinstance(n, dict) for n in nulls):
                problems.append("column-mapped table has an invalid Delta nullCount object")
            else:
                for c in columns:
                    values = [n.get(c["physical"]) if n is not None else None for n in nulls]
                    if any(v is not None and (not isinstance(v, int) or isinstance(v, bool) or v < 0
                           or v > s["numRecords"]) for v, s in zip(values, statistics)):
                        problems.append(f"invalid Delta NULL count for column {c['name']!r}")
                    elif all(v is not None for v in values):
                        result["nullCounts"][c["name"]] = sum(values)
    return result, sorted(set(problems))


def sql_ident(name):
    return "[" + str(name).replace("]", "]]") + "]"


def sql_connection(ws_id, item_id, item_type):
    """The Fabric connector opens SQL analytics endpoints through their parent Lakehouse."""
    if item_type == "SQLEndpoint":
        for lakehouse in fabric_pages(f"/workspaces/{ws_id}/lakehouses"):
            endpoint = (lakehouse.get("properties") or {}).get("sqlEndpointProperties") or {}
            if endpoint.get("id") and str(endpoint["id"]).lower() == str(item_id).lower():
                item_id, item_type = lakehouse["id"], "Lakehouse"
                break
        else:
            raise RuntimeError(f"No parent Lakehouse found for SQL endpoint {item_id} in workspace {ws_id}. "
                               "Check that the restored Lakehouse and its SQL endpoint are ready.")
    return notebookutils.data.connect_to_artifact(item_id, ws_id, item_type)


def sql_query(ws_id, item_id, item_type, text, columns=("n",)):
    """Read-only query against a Warehouse / SQL analytics endpoint (notebookutils data connector).
    The connector returns None, not an empty table, when there are no rows."""
    conn = sql_connection(ws_id, item_id, item_type)
    df = conn.query(text)
    return pd.DataFrame(columns=list(columns)) if df is None else df


def sql_execute(ws_id, item_id, item_type, batches):
    """Run batches through the same Fabric SQL connector used to create the V2_T test data."""
    conn = sql_connection(ws_id, item_id, item_type)
    for batch in batches:
        escaped = batch.replace("'", "''")
        conn.query(f"EXEC(N'{escaped}'); SELECT 1 AS done;")

## Plan
Matches every backed-up data item to the restored item, lists what will be loaded, and prints the DATA ID.

In [ ]:
import csv, io

missing = [n for n in ("SNAPSHOT_ID", "ADLS_ACCOUNT") if "<" in str(globals()[n])]
if missing:
    raise ValueError("Fill in these settings first: " + ", ".join(missing))
if KQL_RESTORE_SOURCE not in ("auto", "parquet", "adx"):
    raise ValueError("KQL_RESTORE_SOURCE must be 'auto', 'parquet' or 'adx'.")
for _n in ("WAREHOUSE_LOAD", "SQL_OBJECTS"):
    if globals()[_n] not in ("script", "run", "off"):
        raise ValueError(f"{_n} must be 'script', 'run' or 'off'.")
if LAKEHOUSE_COPY_METHOD not in ("fastcp", "per_file"):
    raise ValueError("LAKEHOUSE_COPY_METHOD must be 'fastcp' or 'per_file'.")
MANIFEST = load_manifest(SNAPSHOT_ID)
PREFIX = migration_prefix(SNAPSHOT_ID)
SEC = MANIFEST["secondaryBackup"]
if (SEC["account"], SEC["container"]) != (ADLS_ACCOUNT, ADLS_CONTAINER):
    raise ValueError(f"The snapshot's backup run is in {SEC['account']}/{SEC['container']}: use the same ADLS_ACCOUNT "
                     "and ADLS_CONTAINER.")
RUN_FOLDER = f"{SEC['root'].strip('/')}/{SEC['runId']}"
if not adls_exists(f"{RUN_FOLDER}/_run_completed.json"):
    raise RuntimeError(f"Backup run {RUN_FOLDER} has no _run_completed.json: it never finished. Stop.")
BACKUP_LOG = pd.read_csv(io.BytesIO(adls_read(f"{RUN_FOLDER}/backup_log.csv")), dtype=str).fillna("")


def backup_row(item_id, step):
    rows = BACKUP_LOG[(BACKUP_LOG.ItemId.str.lower() == item_id.lower()) & (BACKUP_LOG.Step == step)]
    return rows.iloc[-1].to_dict() if len(rows) else {}


def read_json(path, default):
    try:
        return json.loads(adls_read(path))
    except HttpError as exc:
        if exc.status != 404:
            raise
        return default


STATE = read_json(f"{PREFIX}/data_state.json", {"items": {}})
RESTORE_STATE = read_json(f"{PREFIX}/restore_state.json", {"workspaces": {}})


def save_state():
    adls_write(f"{PREFIX}/data_state.json", json.dumps(STATE, indent=2, sort_keys=True))


def item_state(item):
    return STATE["items"].setdefault(item["id"], {})


known = {w["id"] for w in MANIFEST["workspaces"]}
bad = [k for k in WORKSPACE_MAP if norm_id(k) not in known or not norm_id(WORKSPACE_MAP[k])]
if bad:
    raise ValueError(f"WORKSPACE_MAP entries not in the manifest or not GUIDs: {bad}")
ws_map = {norm_id(k): norm_id(v) for k, v in WORKSPACE_MAP.items()}

STEP_OF = {"Lakehouse": "Data", "Warehouse": "Data", "KQLDatabase": "KQL export", "SQLEndpoint": "SQL scripts"}
SWITCH = {"Lakehouse": RESTORE_LAKEHOUSE_DATA, "KQLDatabase": RESTORE_KQL_DATA,
          "Warehouse": WAREHOUSE_LOAD != "off", "SQLEndpoint": SQL_OBJECTS != "off"}
ITEM_DIR_RE = re.compile(r"^(?P<name>.*)\.(?P<type>[A-Za-z]+)__(?P<id>[0-9a-f-]{36})$")
ITEMS, rows = [], []


def read_data_manifest(raw):
    """[(path, size, status)] from a V2_2 data_manifest.csv. Backups written before 2026-11-17 left the lastModified
    column unquoted ("Tue, 17 Nov 2026 07:35:24 GMT" has commas), so a row can have more than 4 fields: path is the
    first, size the second, status the last."""
    out = []
    rows = csv.reader(io.StringIO(raw.decode("utf-8-sig")))
    next(rows, None)                                     # header
    for row in rows:
        if not row:
            continue
        if len(row) < 4 or not str(row[1]).strip().isdigit():
            raise ValueError(f"data_manifest.csv: can't read the row {row[:4]}")
        out.append((row[0], int(row[1]), row[-1].strip()))
    return out


def check_manifest_files(it, files):
    """The backup's data_manifest.csv lists every file it copied: each must still be in ADLS with that size."""
    if not any(rel == "data_manifest.csv" for rel, _, _ in files):
        return ["data_manifest.csv is missing in the backup"]
    man = read_data_manifest(adls_read(f"{it['folder']}/data_manifest.csv"))
    have = dict(it["data"])
    bad = [path for path, size, status in man if status == "copied" and have.get(path) != size]
    failed = sum(1 for _, _, status in man if status != "copied")
    out = []
    if bad:
        out.append(f"{len(bad)} file(s) listed in data_manifest.csv are missing or changed in ADLS, first: {bad[0]}")
    if failed:
        out.append(f"{failed} file(s) failed during the backup")
    return out


def lakehouse_layout_problem(it):
    """The backup's table layout (lakehouse.json, or the table folders) must match the restored Lakehouse:
    schema-enabled = Tables/<schema>/<table>, otherwise Tables/<table>. Returns (problem or "", text)."""
    try:
        lay = json.loads(adls_read(f"{it['folder']}/lakehouse.json"))
    except HttpError as exc:
        if exc.status != 404:
            raise
        tables = sorted({table_root(r) for r, _ in it["data"] if table_root(r)})
        deep = [t for t in tables if t.count("/") >= 2]
        lay = {"schemaEnabled": bool(tables) and len(deep) == len(tables), "defaultSchema": "", "tables": tables,
               "schemas": sorted({t.split("/")[1] for t in deep}), "inferred": True}
    props = fabric("GET", f"/workspaces/{it['ws']}/lakehouses/{it['target']}").get("properties") or {}
    target_ds = props.get("defaultSchema") or ""
    text = (f"schemas: {', '.join(lay['schemas']) or '-'}" if lay["schemaEnabled"] else "no schemas")
    if not lay["tables"]:
        return "", text
    if lay["schemaEnabled"] and not target_ds:
        return ("the backup is from a schema-enabled Lakehouse (tables in Tables/<schema>/<table>) but the restored "
                "Lakehouse has no schemas: its tables would not be readable. Recreate it with schemas enabled (its Git "
                "definition lakehouse.metadata.json must have defaultSchema), then run again"), text
    if not lay["schemaEnabled"] and target_ds:
        return ("the backup is from a Lakehouse without schemas (Tables/<table>) but the restored Lakehouse is "
                f"schema-enabled (default schema {target_ds}): its tables would not be readable. Recreate it without "
                "schemas, then run again"), text
    return "", text


def plan_lakehouse(it):
    problem, layout = lakehouse_layout_problem(it)
    if problem:
        it["problems"].append(problem)
        return "BLOCKED", problem, [problem]
    st = item_state(it)
    target = dict(onelake_list(it["ws"], it["target"], "Files") + onelake_list(it["ws"], it["target"], "Tables"))
    ours = set(st.get("tables", []))
    in_target = {table_root(r) for r in target if table_root(r)}
    skip = sorted(t for t in {table_root(r) for r, _ in it["data"] if table_root(r)} if t in in_target and t not in ours)
    todo, there, conflicts = [], 0, []
    for rel, size in it["data"]:
        if table_root(rel) in skip:
            continue
        if rel in target:
            if target[rel] == size:
                there += 1
                continue
            if not (st.get("started") and (table_root(rel) in ours or rel.startswith("Files/"))):
                conflicts.append(rel)
                continue
        todo.append((rel, size))
    it.update(todo=todo, skipTables=skip, conflicts=conflicts)
    detail = f"{len(todo)} file(s), {human(sum(s for _, s in todo))} to copy; {there} already there; {layout}"
    if skip:
        detail += f" | tables that already hold data in the target, left alone: {', '.join(skip)}"
    if conflicts:
        detail += f" | {len(conflicts)} file(s) exist with different content, left alone: {conflicts[0]}"
    return ("load" if todo else "nothing to load"), detail, [len(todo), skip, conflicts]


def plan_kql(it):
    props = fabric("GET", f"/workspaces/{it['ws']}/kqlDatabases/{it['target']}").get("properties") or {}
    it["uri"], it["db"] = props.get("queryServiceUri"), props.get("databaseName") or it["displayName"]
    if not it["uri"]:
        raise HttpError(None, "NoQueryUri", "no query URI for the restored KQL database", it["target"])
    folders = defaultdict(list)
    if it["source"] == "adx":     # one entry per table: the source table name in ADX
        for t in it["adx"].get("tables") or {}:
            folders[safe_name(t)].append(t)
    else:
        for rel, size in it["data"]:
            if "/" in rel and rel.lower().endswith(".parquet"):
                folders[rel.split("/")[0]].append(rel)
    tables = {safe_name(t["TableName"]): t["TableName"] for t in kusto_mgmt(it["uri"], it["db"], ".show tables")}
    st = item_state(it)
    busy, todo = [], {}
    for f, rels in sorted(folders.items()):
        t = tables.get(f)
        if t and t in st.get("done", {}):
            continue
        if t and t not in st.get("done", {}) and t not in st.get("started", []) and kusto_count(it["uri"], it["db"], t):
            busy.append(t)
            continue
        todo[f] = sorted(rels)
    it.update(folders=todo, busy=busy, missingTables=[f for f in todo if f not in tables])
    expected = it["backupRows"]
    detail = (f"{len(todo)} table(s) from ADX {it['adx'].get('cluster')} / {it['adx'].get('database')}"
              if it["source"] == "adx" else
              f"{len(todo)} table(s), {sum(len(v) for v in todo.values())} Parquet file(s)") + \
        (f", {int(float(expected)):,} rows backed up" if expected else "")
    if it["missingTables"]:
        detail += f" | {len(it['missingTables'])} table(s) not in the database: the backed-up schema script runs first"
    if busy:
        detail += f" | tables that already hold rows, left alone: {', '.join(busy)}"
    return ("ingest" if todo else "nothing to load"), detail, [sorted(todo), busy, it["missingTables"]]


def plan_warehouse(it):
    tables = sorted({table_root(r) for r, _ in it["data"] if table_root(r)})
    it["tables"] = {}
    manual = []
    for t in tables:
        res, problems = delta_active_files(it["folder"], t, it["data"])
        if res is None or problems:
            manual.append(f"{t}: {'; '.join(problems)}")
        else:
            it["tables"][t] = res
    it["manual"] = manual
    rows_ = sum(r["rows"] or 0 for r in it["tables"].values())
    detail = f"{len(it['tables'])} table(s), {rows_:,} rows by the Delta log; mode '{WAREHOUSE_LOAD}'"
    mapped = sum(bool(r.get("columns")) for r in it["tables"].values())
    if mapped:
        detail += f" | {mapped} column-mapped table(s): typed OPENROWSET, physical to logical columns"
    if manual:
        detail += f" | {len(manual)} table(s) need loading by hand: {manual[0]}"
    return (f"COPY INTO ({WAREHOUSE_LOAD})" if it["tables"] else "manual load" if manual else "nothing to load"), detail, \
        [it["folder"], sorted(it["tables"].items()), manual]


def plan_sql(it):
    return (f"SQL objects ({SQL_OBJECTS})" if it["sql"] else "nothing to load"), \
        f"{len(it['sql'])} view(s)/function(s)/procedure(s); mode '{SQL_OBJECTS}'", [sorted(it["sql"])]


PLANNERS = {"Lakehouse": plan_lakehouse, "KQLDatabase": plan_kql, "Warehouse": plan_warehouse, "SQLEndpoint": plan_sql}

for mws in MANIFEST["workspaces"]:
    if not mws["ready"]:
        print(f"{mws['name']}: skipped (not READY in the manifest)")
        continue
    old = mws["id"]
    rst = RESTORE_STATE["workspaces"].get(old, {})
    new, how = resolve_target_workspace(old, mws["name"], ws_map.get(old), rst.get("target"))
    ws_problems, now = ([] if new else [how]), {}
    new = new or old
    if how != "same ID" and new:
        print(f"{mws['name']}: target workspace {new} ({how})")
    if REQUIRE_GIT_RESTORE_DONE and not RESTORE_STATE["workspaces"].get(old, {}).get("updated"):
        ws_problems.append("step 5 (V2_5_Restore_From_Git) hasn't finished for this workspace (set REQUIRE_GIT_RESTORE_DONE = False only if "
                           "the items were recreated some other way)")
    try:
        now = {key_of(i["type"], i["displayName"]): i for i in workspace_items(new)}
    except HttpError as exc:
        ws_problems.append(f"can't read the target workspace {new} ({exc.short()})")
    folder = mws.get("secondaryFolder") or ""
    backed = {}
    for d, _, is_dir in (adls_list(folder, recursive=False) if folder else []):
        m = ITEM_DIR_RE.match(d)
        if is_dir and m:
            backed[m.group("id").lower()] = f"{folder}/{d}"
    for mi in mws["items"]:
        typ = mi["type"]
        if typ not in PLANNERS or is_dataflow_staging(mi):     # Dataflow Gen2 staging: nothing backed up, not restored
            continue
        it = {"workspace": mws["name"], "wsOld": old, "ws": new, "id": mi["id"].lower(), "type": typ,
              "displayName": mi["displayName"], "folder": backed.get(mi["id"].lower()), "problems": list(ws_problems),
              "target": (now.get(key_of(typ, mi["displayName"])) or {}).get("id")}
        step = STEP_OF[typ]
        if typ == "KQLDatabase":
            step = {"parquet": "KQL export", "adx": KQL_ADX_STEP}.get(KQL_RESTORE_SOURCE) or data_step(
                typ, lambda s, i=mi["id"]: backup_row(i, s).get("Status", ""))[0]
        it["step"], it["source"] = step, "adx" if step == KQL_ADX_STEP else "adls"
        b = backup_row(mi["id"], step)
        it["backupStatus"], it["backupRows"] = b.get("Status", ""), b.get("Rows", "")
        files = adls_list(it["folder"]) if it["folder"] else []
        it["data"] = [(rel[5:], size) for rel, size, d in files if not d and rel.startswith("data/")]
        it["sql"] = [rel for rel, _, d in files if not d and rel.startswith("sql/") and rel.endswith(".sql")]
        action, detail, sig = "off", f"switched off ({'RESTORE_LAKEHOUSE_DATA' if typ == 'Lakehouse' else 'RESTORE_KQL_DATA' if typ == 'KQLDatabase' else 'WAREHOUSE_LOAD' if typ == 'Warehouse' else 'SQL_OBJECTS'})", []
        if SWITCH[typ]:
            if not it["folder"]:
                it["problems"].append(f"no folder for this item in backup run {SEC['runId']}")
            if it["backupStatus"] != "OK":
                it["problems"].append(f"backup step '{step}' was {it['backupStatus'] or 'not recorded'}")
            if it["source"] == "adx" and it["folder"]:
                it["adx"] = read_json(f"{it['folder']}/kusto_copy/_destination.json", None)
                if not it["adx"]:
                    it["problems"].append("the backup has no kusto_copy/_destination.json: can't tell where the "
                                          "rows are in ADX")
            if typ in ("Lakehouse", "Warehouse") and it["folder"]:
                it["problems"] += check_manifest_files(it, files)
            if not it["target"]:
                it["problems"].append("not found in the target workspace: restore the item first (step 5, V2_5)")
            if not it["problems"]:
                try:
                    action, detail, sig = PLANNERS[typ](it)
                except HttpError as exc:
                    it["problems"].append(f"can't read the restored item ({exc.short()})")
        it["action"] = "BLOCKED" if it["problems"] else action
        it["detail"] = " | ".join(it["problems"]) if it["problems"] else detail
        it["sig"] = sig
        ITEMS.append(it)
        rows.append({"Workspace": it["workspace"], "Item": it["displayName"], "Type": typ, "Backup": it["backupStatus"],
                     "Target ID": it["target"] or "", "Action": it["action"], "Detail": it["detail"]})

PLAN_DF = pd.DataFrame(rows, columns=["Workspace", "Item", "Type", "Backup", "Target ID", "Action", "Detail"])
show(PLAN_DF)
TODO = [i for i in ITEMS if i["action"] not in ("BLOCKED", "off", "nothing to load")]
DATA_ID = plan_hash({"snapshot": SNAPSHOT_ID, "run": SEC["runId"], "modes": [WAREHOUSE_LOAD, SQL_OBJECTS],
                     "items": [(i["id"], i["ws"], i["target"], i["action"], i["sig"]) for i in ITEMS]})
blocked = [i for i in ITEMS if i["action"] == "BLOCKED"]
print(f"\n{len(TODO)} item(s) to load, {len(blocked)} BLOCKED. DATA ID: {DATA_ID}")
if blocked:
    print("BLOCKED items are not loaded. Fix them and run the dry run again, or restore them by hand.")
if DRY_RUN:
    PROCEED = False
    print(f"DRY_RUN = True: nothing loaded. To load, set DRY_RUN = False and CONFIRM_DATA_ID = \"{DATA_ID}\".")
elif CONFIRM_DATA_ID != DATA_ID:
    raise RuntimeError(f"CONFIRM_DATA_ID doesn't match this plan's DATA ID ({DATA_ID}). Something changed since the "
                       "dry run, or the ID wasn't pasted. Review the plan above and confirm again.")
else:
    PROCEED = True

## Lakehouse data
Copies `Files/` and `Tables/` back into OneLake byte for byte (Delta history kept): data files first, Delta logs last, so a table only appears once all its files are there. Then checks every file.
With `LAKEHOUSE_COPY_METHOD = "fastcp"` each table folder is copied in bulk by azcopy (`notebookutils.fs.fastcp`), after a tiny test copy in ADLS that checks how fastcp lays out folders. Anything fastcp didn't place is copied file by file. Nothing that is already in the Lakehouse is replaced, and the backup in ADLS is never moved or changed.

In [ ]:
def copy_per_file(it, files):
    """The original copy: every file through this notebook, MAX_PARALLEL_FILES at a time. Returns (copied, there,
    bytes, errors)."""
    copied, there, nbytes, errors = 0, 0, 0, []
    with ThreadPoolExecutor(MAX_PARALLEL_FILES) as pool:
        futs = {pool.submit(restore_file, f"{it['folder']}/data/{rel}", size, it["ws"], it["target"], rel, True):
                (rel, size) for rel, size in files}
        for fut in as_completed(futs):
            rel, size = futs[fut]
            try:
                if fut.result() == "already there":
                    there += 1
                else:
                    copied += 1
                nbytes += size
            except HttpError as exc:
                errors.append(f"{rel}: {exc.short()}")
    return copied, there, nbytes, errors


def bulk_units(files, all_files):
    """Folders fastcp can copy whole: each Delta table folder, plus Files/ when no Delta table lives under it.
    Returns ({unit: [(rel, size)]}, [(rel, size) left for the per-file copy])."""
    roots = {table_root(r) for r, _ in all_files if table_root(r)}      # tables known from their _delta_log
    files_has_tables = any(r.startswith("Files/") for r in roots)
    units, rest = {}, []
    for rel, size in files:
        root = next((r for r in roots if rel.startswith(r + "/")), None)
        if root:
            units.setdefault(root, []).append((rel, size))
        elif rel.startswith("Files/") and not files_has_tables:
            units.setdefault("Files", []).append((rel, size))
        else:
            rest.append((rel, size))
    return units, rest


def copy_bulk(it, units, delta_logs):
    """fastcp each unit folder from the backup into the Lakehouse (existing files are never replaced). Phase 1
    (delta_logs False) leaves every _delta_log out; phase 2 copies only the _delta_log folders. Returns the error
    texts; files it didn't place are picked up by the per-file copy afterwards."""
    errors = []
    for unit in sorted(units):
        src_rel = unit + ("/_delta_log" if delta_logs else "")
        parent = src_rel.rsplit("/", 1)[0] if "/" in src_rel else ""
        dst_rel = parent if FASTCP_LAYOUT == "into" else src_rel
        print(f"  fastcp {src_rel} ({len(units[unit])} file(s), {human(sum(s for _, s in units[unit]))})")
        t0 = time.time()
        try:
            if dst_rel:
                onelake_mkdir(it["ws"], it["target"], dst_rel)
            fastcp(abfss_adls(f"{it['folder']}/data/{src_rel}"), abfss_onelake(it["ws"], it["target"], dst_rel),
                   exclude_delta_log=not delta_logs and unit != "Files")
            print(f"    done in {time.time() - t0:,.0f} s")
        except Exception as exc:
            errors.append(f"{src_rel}: {exc.short() if isinstance(exc, HttpError) else str(exc)[:300]}")
    return errors


def load_lakehouse(it):
    st = item_state(it)
    st["started"] = True
    st["tables"] = sorted(set(st.get("tables", [])) | {table_root(r) for r, _ in it["todo"] if table_root(r)})
    save_state()
    first = [f for f in it["todo"] if "/_delta_log/" not in f[0]]
    last = sorted(f for f in it["todo"] if "/_delta_log/" in f[0])
    method = LAKEHOUSE_COPY_METHOD
    if method == "fastcp":
        layout, why = fastcp_layout()
        if not layout:
            method = "per_file"
            log(it["workspace"], it, "Lakehouse data", "Info", f"fastcp not usable here ({why}); copying file by file")
    copied, there, nbytes, errors, bulk = 0, 0, 0, [], 0
    for phase, batch in ((1, first), (2, last)):
        if errors and phase == 2:
            errors.append("Delta logs not copied because data files failed: the tables stay invisible, not corrupt. "
                          "Run again.")
            break
        todo = batch
        if method == "fastcp" and batch:
            units, rest = bulk_units(batch, it["data"])
            if phase == 2:
                units = {u: f for u, f in units.items() if u != "Files"}
            bulk_errors = copy_bulk(it, units, delta_logs=phase == 2)
            now = dict(onelake_list(it["ws"], it["target"], "Files") + onelake_list(it["ws"], it["target"], "Tables"))
            rest_set = set(rest)
            placed = [(r, s) for r, s in batch if now.get(r) == s and (r, s) not in rest_set]
            bulk += len(placed)
            nbytes += sum(s for _, s in placed)
            placed_set = set(placed)
            todo = [f for f in batch if f not in placed_set]
            if bulk_errors:
                log(it["workspace"], it, "Lakehouse data", "Info",
                    f"fastcp failed for {len(bulk_errors)} folder(s), those files are copied one by one: {bulk_errors[0]}")
        c, t, b, e = copy_per_file(it, todo)
        copied, there, nbytes, errors = copied + c, there + t, nbytes + b, errors + e
    how = (f"{bulk} by fastcp, {copied} file by file" if method == "fastcp" else f"{copied} copied")
    log(it["workspace"], it, "Lakehouse data", "OK" if not errors else ("Partial" if copied or bulk else "Failed"),
        f"{how}, {there} already there, {human(nbytes)}" +
        (f" | {len(errors)} error(s), first: {errors[0]}" if errors else ""))

for it in [i for i in TODO if i["type"] == "Lakehouse"] if PROCEED else []:
    print(f"Lakehouse data: {it['workspace']} / {it['displayName']} ({len(it['todo'])} files)")
    try:
        load_lakehouse(it)
    except HttpError as exc:
        log(it["workspace"], it, "Lakehouse data", "Failed", exc.short())
if not PROCEED:
    print("Skipped (dry run).")

## KQL database data
Ingests each table's Parquet export from ADLS. Update policies are paused during ingestion (so rows aren't copied twice into tables that are also restored) and always put back.

In [ ]:
def load_kql(it):
    uri, db, st = it["uri"], it["db"], item_state(it)
    if it["missingTables"]:
        schema = f"{it['folder']}/database_schema.kql"
        if not adls_exists(schema):
            raise HttpError(None, "NoSchema", f"tables {it['missingTables']} are missing and the backup has no schema "
                            "script", schema)
        kusto_mgmt(uri, db, ".execute database script with (ContinueOnErrors=false) <|\n" +
                   adls_read(schema).decode("utf-8"))
    tables = {safe_name(t["TableName"]): t["TableName"] for t in kusto_mgmt(uri, db, ".show tables")}
    done = st.setdefault("done", {})
    started = st.setdefault("started", [])
    saved = st.setdefault("policies", {})
    if not saved:
        for row in kusto_mgmt(uri, db, ".show table * policy update"):
            pol = row.get("Policy")
            if pol and pol not in ("[]", "null"):
                saved[row["EntityName"].split("].[")[-1].strip("[]")] = pol
        save_state()
    errors = []
    try:
        for t in saved:
            kusto_mgmt(uri, db, f".delete table [{kql_string(t)}] policy update")
        for folder, rels in sorted(it["folders"].items()):
            table = tables.get(folder)
            if not table:
                errors.append(f"{folder}: no matching table in the restored database")
                continue
            if table in done:
                continue
            current = kusto_count(uri, db, table)
            if current:
                errors.append(f"{table}: an earlier run was interrupted during ingestion and the table has {current:,} "
                              f"rows. Check it, run '.clear table [{table}] data' and run again." if table in started
                              else f"{table}: already has {current:,} rows; not touched")
                continue
            started.append(table)
            save_state()
            if it["source"] == "adx":
                src = (f"cluster({kql_string(it['adx']['cluster'])}).database({kql_string(it['adx']['database'])})"
                       f".[{kql_string(rels[0])}]")
                op = kusto_mgmt(uri, db, f".set-or-append async [{kql_string(table)}] <| {src}",
                                retry=False)[0]["OperationId"]
                kusto_wait(uri, db, op, f"copy {table} from ADX")
                rels = []
            for i in range(0, len(rels), 50):
                sources = ", ".join(f"h'https://{ADLS_ACCOUNT}.blob.core.windows.net/{ADLS_CONTAINER}/"
                                    f"{quote(it['folder'] + '/data/' + r, safe='/')};impersonate'" for r in rels[i:i + 50])
                op = kusto_mgmt(uri, db, f".ingest async into table [{kql_string(table)}] ({sources}) "
                                         "with (format='parquet')", retry=False)[0]["OperationId"]
                kusto_wait(uri, db, op, f"ingest {table}")
            done[table] = kusto_count(uri, db, table)
            save_state()
            print(f"    {table}: {done[table]:,} rows")
    finally:
        for t, pol in saved.items():
            kusto_mgmt(uri, db, f".alter table [{kql_string(t)}] policy update {kql_string(pol)}")
        if saved:
            st["policies"] = {}
            save_state()
    log(it["workspace"], it, "KQL data", "OK" if not errors else ("Partial" if done else "Failed"),
        f"{len(done)} table(s), {sum(done.values()):,} rows" + (f"; update policies of {len(saved)} table(s) paused "
        "and put back" if saved else "") + (f" | {len(errors)} problem(s), first: {errors[0]}" if errors else ""))


for it in [i for i in TODO if i["type"] == "KQLDatabase"] if PROCEED else []:
    print(f"KQL data: {it['workspace']} / {it['displayName']}")
    try:
        load_kql(it)
    except HttpError as exc:
        log(it["workspace"], it, "KQL data", "Failed", exc.short())
if not PROCEED:
    print("Skipped (dry run).")

## Warehouse tables and SQL endpoint objects
Writes a `.sql` script per item to ADLS (`_migration/<snapshot>/data/scripts/`). With `"run"` it also runs it here: a Warehouse table is loaded only while it is empty, and SQL objects that already exist are skipped.

In [ ]:
SCRIPTS = f"{PREFIX}/data/scripts"


def wh_table_name(root):
    parts = root.split("/")[1:]                      # Tables/<schema>/<table> or Tables/<table>
    return sql_ident(parts[0] if len(parts) > 1 else "dbo") + "." + sql_ident(parts[-1])


def wh_batches(it, root, res):
    base = f"https://{ADLS_ACCOUNT}.blob.core.windows.net/"
    urls = [base + quote(f"{ADLS_CONTAINER}/{it['folder']}/data/{root}/{p}", safe="/") for p in res["files"]]
    columns = res.get("columns")
    if columns:
        target = ", ".join(sql_ident(c["name"]) for c in columns)
        source = ", ".join("[src]." + sql_ident(c["physical"]) for c in columns)
        schema = ", ".join(sql_ident(c["physical"]) + " " + c["type"] for c in columns)
        # Fabric binds Parquet fields by their physical names, not renamed WITH column paths.
        selects = [f"SELECT {source}\nFROM OPENROWSET(BULK '{u}', FORMAT = 'PARQUET')\n"
                   f"WITH ({schema}) AS [src]" for u in urls]
        return [f"INSERT INTO {wh_table_name(root)} ({target})\n" + "\nUNION ALL\n".join(selects[i:i + 100]) + ";"
                for i in range(0, len(selects), 100)]
    return [f"COPY INTO {wh_table_name(root)}\nFROM " + ",\n    ".join("'" + u + "'" for u in urls[i:i + 100]) +
            "\nWITH (FILE_TYPE = 'PARQUET');" for i in range(0, len(urls), 100)]


def wh_count(it, root):
    df = sql_query(it["ws"], it["target"], "Warehouse", f"SELECT COUNT_BIG(*) AS n FROM {wh_table_name(root)}")
    return int(df.iloc[0, 0])


def load_warehouse(it):
    out = [f"-- Load the backed-up tables into the restored Warehouse '{it['displayName']}' (backup {SEC['runId']}).",
           "-- Run as a user with Storage Blob Data Reader on the backup container. Load only EMPTY tables.", ""]
    for root, res in sorted(it["tables"].items()):
        out += wh_batches(it, root, res) if res["files"] else [f"-- {wh_table_name(root)}: empty, nothing to load"]
        out.append(f"SELECT COUNT_BIG(*) AS rows_now, {res['rows'] if res['rows'] is not None else 'NULL'} AS "
                   f"rows_expected FROM {wh_table_name(root)};\n")
    for m in it["manual"]:
        out.append(f"-- NOT SCRIPTED, load by hand: {m}")
    path = f"{SCRIPTS}/{safe_name(it['workspace'])}/{safe_name(it['displayName'])}.Warehouse_load.sql"
    adls_write(path, "\n".join(out) + "\n")
    if WAREHOUSE_LOAD != "run":
        log(it["workspace"], it, "Warehouse data", "Manual step", f"run {path} in the Warehouse ({len(it['tables'])} "
            "tables)" + (f"; {len(it['manual'])} table(s) by hand" if it["manual"] else ""))
        return
    st = item_state(it)
    done, started, errors = st.setdefault("done", {}), st.setdefault("started", []), []
    for root, res in sorted(it["tables"].items()):
        if root in done or not res["files"]:
            continue
        try:
            current = wh_count(it, root)
            if current:
                errors.append(f"{root}: has {current:,} rows " + ("after an interrupted load: check it, TRUNCATE it and "
                              "run again" if root in started else "already; not touched"))
                continue
            started.append(root)
            save_state()
            sql_execute(it["ws"], it["target"], "Warehouse", wh_batches(it, root, res))
            done[root] = wh_count(it, root)
            save_state()
            print(f"    {root}: {done[root]:,} rows")
        except Exception as exc:                   # ODBC and connector errors have several types
            errors.append(f"{root}: {str(exc)}")
    errors += [f"by hand: {m}" for m in it["manual"]]
    log(it["workspace"], it, "Warehouse data", "OK" if not errors else ("Partial" if done else "Failed"),
        f"{len(done)}/{len(it['tables'])} table(s) loaded; script {path}" +
        (f" | {len(errors)} problem(s), first: {errors[0]}" if errors else ""))


def load_sql_objects(it):
    parts = [f"-- Views, functions and procedures of '{it['displayName']}' (backup {SEC['runId']}).",
             "-- Run in the restored SQL analytics endpoint after its tables have appeared there.", ""]
    bodies = {}
    for rel in sorted(it["sql"]):
        bodies[rel] = adls_read(f"{it['folder']}/{rel}").decode("utf-8").rstrip()
        parts += [f"-- {rel}", bodies[rel], "GO", ""]
    path = f"{SCRIPTS}/{safe_name(it['workspace'])}/{safe_name(it['displayName'])}.SQLEndpoint_objects.sql"
    adls_write(path, "\n".join(parts))
    if SQL_OBJECTS != "run":
        log(it["workspace"], it, "SQL objects", "Manual step", f"run {path}")
        return
    existing = set(sql_query(it["ws"], it["target"], "SQLEndpoint",
                             "SELECT s.name + '/' + o.name AS k FROM sys.objects o JOIN sys.schemas s "
                             "ON s.schema_id = o.schema_id WHERE o.is_ms_shipped = 0", columns=["k"]).iloc[:, 0])
    todo = {r: b for r, b in bodies.items() if f"{r.split('/')[1]}/{r.rsplit('/', 1)[1][:-4]}" not in existing}
    errors = {}
    for _ in range(3):                             # objects can depend on each other: retry the failures
        for rel in list(todo):
            try:
                sql_execute(it["ws"], it["target"], "SQLEndpoint", [todo[rel]])
                todo.pop(rel)
                errors.pop(rel, None)
            except Exception as exc:
                errors[rel] = str(exc)
        if not todo:
            break
    log(it["workspace"], it, "SQL objects", "OK" if not todo else "Partial",
        f"{len(bodies) - len(todo)}/{len(bodies)} object(s) present; script {path}" +
        (f" | {len(todo)} failed, first: {next(iter(errors.items()))}" if todo else ""))


for it in [i for i in TODO if i["type"] in ("Warehouse", "SQLEndpoint")] if PROCEED else []:
    print(f"{it['type']}: {it['workspace']} / {it['displayName']}")
    try:
        (load_warehouse if it["type"] == "Warehouse" else load_sql_objects)(it)
    except Exception as exc:
        log(it["workspace"], it, "Warehouse data" if it["type"] == "Warehouse" else "SQL objects", "Failed",
            exc.short() if isinstance(exc, HttpError) else str(exc))
if not PROCEED:
    print("Skipped (dry run).")

## Verify
Reads every restored item again and compares it with the backup: files and sizes, row counts, objects. Saves the log and the result in ADLS.

In [ ]:
def verify(it):
    st = item_state(it)
    if it["type"] == "Lakehouse":
        now = dict(onelake_list(it["ws"], it["target"], "Files") + onelake_list(it["ws"], it["target"], "Tables"))
        want = [(r, s) for r, s in it["data"] if table_root(r) not in it["skipTables"] and r not in it["conflicts"]]
        bad = [r for r, s in want if now.get(r) != s]
        left = len(it["skipTables"]) + len(it["conflicts"])
        return ("OK" if not bad and not left else "Failed" if bad else "Partial",
                f"{len(want) - len(bad)}/{len(want)} backed-up files present with the same size"
                + (f"; missing or different: {bad[0]}" if bad else "")
                + (f"; {left} table(s)/file(s) left alone (already in the target)" if left else ""))
    if it["type"] == "KQLDatabase":
        tables = {safe_name(t["TableName"]): t["TableName"] for t in kusto_mgmt(it["uri"], it["db"], ".show tables")}
        got = {f: kusto_count(it["uri"], it["db"], tables[f]) if f in tables else None for f in it["folders"]}
        total = sum(v or 0 for v in got.values()) + sum(v for t, v in st.get("done", {}).items()
                                                       if safe_name(t) not in got)
        expected = int(float(it["backupRows"])) if it["backupRows"] else None
        ok = expected is not None and total == expected and not it["busy"]
        return ("OK" if ok else "Failed" if not total else "Partial",
                f"{total:,} rows now" + (f", {expected:,} backed up" if expected is not None else ", backup count unknown")
                + (f"; tables left alone: {', '.join(it['busy'])}" if it["busy"] else ""))
    if it["type"] == "Warehouse":
        bad, empty, null_checks = [], 0, 0
        for root, res in sorted(it["tables"].items()):
            nulls = res.get("nullCounts") or {}
            if nulls:
                expressions = ["COUNT_BIG(*) AS n"] + [f"COUNT_BIG({sql_ident(c)}) AS c{j}"
                                                       for j, c in enumerate(nulls)]
                df = sql_query(it["ws"], it["target"], "Warehouse",
                               "SELECT " + ", ".join(expressions) + f" FROM {wh_table_name(root)}")
                n = int(df.iloc[0, 0])
            else:
                n = wh_count(it, root)
            empty += n == 0 and bool(res["rows"])
            differences = []
            if res["rows"] is not None and n != res["rows"]:
                differences.append(f"{n:,} rows, Delta log says {res['rows']:,}")
            for j, (column, expected) in enumerate(nulls.items()):
                actual = n - int(df.iloc[0, j + 1])
                null_checks += 1
                if actual != expected:
                    differences.append(f"{column}: {actual:,} NULLs, Delta log says {expected:,}")
            if differences:
                bad.append(f"{root}: " + "; ".join(differences))
        if WAREHOUSE_LOAD != "run" and bad and empty == len(bad):
            return "Manual step", (f"{empty} table(s) still empty: run the load script, then run this notebook again "
                                   "to check the row counts")
        return ("OK" if not bad and not it["manual"] else "Partial" if not bad else "Failed",
                f"{len(it['tables']) - len(bad)}/{len(it['tables'])} table(s) match the backed-up row count"
                + (f"; {null_checks} backed-up column NULL count(s) checked" if null_checks else "")
                + (f"; {bad[0]}" if bad else "") + (f"; {len(it['manual'])} table(s) by hand" if it["manual"] else ""))
    return None


if PROCEED:
    print("Checking every loaded item against the backup...")
    for it in TODO:
        try:
            res = verify(it)
        except Exception as exc:
            res = ("Failed", exc.short() if isinstance(exc, HttpError) else str(exc))
        if res:
            log(it["workspace"], it, "Verify", *res)
    df = save_log(PREFIX, "data_log")
    result = {"snapshot": SNAPSHOT_ID, "run": SEC["runId"], "dataId": DATA_ID, "finishedUtc": NOW_ID,
              "items": {i["id"]: {"name": i["displayName"], "type": i["type"], "action": i["action"]} for i in ITEMS},
              "verify": df[df.Step == "Verify"][["ItemId", "Status", "Detail"]].to_dict("records")}
    adls_write(f"{PREFIX}/data_result_{NOW_ID}.json", json.dumps(result, indent=2))
    attention = df[~df.Status.isin(["OK", "Info"])]
    show(df.pivot_table(index="Step", columns="Status", values="Item", aggfunc="count", fill_value=0))
    blocked = [i for i in ITEMS if i["action"] == "BLOCKED"]
    if attention.empty and not blocked:
        print("\nALL DATA VERIFIED: every loaded item matches its backup.")
    else:
        print(f"\n{len(attention)} row(s) need attention, {len(blocked)} item(s) BLOCKED (Manual step rows are "
              "expected work):")
        show(attention)
        print("Fix the cause and run again (new dry run + DATA ID): finished tables and files are never loaded twice.")
    print(f"Log: {ADLS_CONTAINER}/{PREFIX}/data_log_{NOW_ID}.csv")